<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/wm_gguf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu130 \
  --no-cache-dir

## 5M-WM

In [4]:
#!/usr/bin/env python3
"""
SOMALA • Closed-Loop Multi-Step 5-Model Selective GGUF World Model (VRAM in GB)
Trajectory: s_0 -> rollout/policy/SQL -> s_1 = step_fn(s_0, a_0) -> s_2 -> ...
"""

import argparse
import gc
import hashlib
import json
import math
import re
import subprocess
import time
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
from huggingface_hub import HfApi, hf_hub_download
from llama_cpp import Llama

ANCHORS: List[int] = [2, 3, 5, 7, 11, 13]

SELECTIVE_MODELS: Dict[str, str] = {
    "physical": "frankmorales2020/nemotron-3.5-bolt-e4b-topo-governed-fineweb-gguf",
    "perceive": "frankmorales2020/Gemma-4-26B-A4B-alpaca-topo-epoch-1-Q8_0-GGUF",
    "simulate": "frankmorales2020/muse-glimmer-30b-topo-governed-cbp-fineweb-gguf",
    "policy": "frankmorales2020/deepseek-moe-16b-topo-cbp-merged-gguf",
    "act": "frankmorales2020/mistral-7b-text-to-sql-topo-Q8_0-GGUF",
}

VALID_ACTIONS: List[str] = ["NO_OP", "THROTTLE_20", "COOLING_BOOST", "SHUTDOWN"]

ACTION_COSTS: Dict[str, float] = {
    "NO_OP": 0.0,
    "THROTTLE_20": 2.0,
    "COOLING_BOOST": 3.0,
    "SHUTDOWN": 18.0,
}

_RESIDENT_MODELS: Dict[str, Llama] = {}


@dataclass
class State:
    temp_c: float
    vib_rms: float
    rate_c_min: float
    load_pct: float


def format_observation(s: State, asset: str = "Bearing B4") -> str:
    return (
        f"{asset}: temp={s.temp_c:.2f}C, vib={s.vib_rms:.3f} rms, "
        f"dT/dt={s.rate_c_min:+.3f}C/min, load={s.load_pct:.1f}%."
    )


def format_vram_gb(mib: Optional[float | int]) -> str:
    if mib is None:
        return "N/A"
    return f"{float(mib) / 1024.0:.2f} GB"


def get_gguf(repo_id: str, cache_dir: Optional[Path] = None) -> Path:
    api = HfApi()
    files = [f for f in api.list_repo_files(repo_id) if f.lower().endswith(".gguf")]
    if not files:
        raise FileNotFoundError(f"No .gguf file found in {repo_id}")
    return Path(
        hf_hub_download(
            repo_id=repo_id,
            filename=files[0],
            cache_dir=str(cache_dir) if cache_dir else None,
        )
    )


def check_drift(path: Path) -> Tuple[str, float, float]:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(chunk)
        rows = []
        for p in ANCHORS:
            f.seek(p * 4096)
            buf = f.read(64)
            if not buf:
                continue
            rows.append(np.frombuffer(buf, dtype=np.uint8).astype(np.float64))
    if not rows:
        raise ValueError(f"Unable to read anchor slices from {path}")
    mat = np.vstack(rows)
    ref = mat.copy()
    drift = float(np.max(np.abs(mat - ref)))
    var = float(np.var(mat - ref))
    if not (math.isfinite(drift) and math.isfinite(var)):
        raise ValueError("Non-finite drift/variance metric computed.")
    return h.hexdigest()[:16], drift, var


def step_fn(s: State, action: str, dt: float = 1.0) -> State:
    if action == "NO_OP":
        target_rate = s.rate_c_min + 0.15 * max(0.0, (s.load_pct - 70.0) / 30.0)
        rate = s.rate_c_min + 0.5 * (target_rate - s.rate_c_min)
        return State(
            temp_c=s.temp_c + rate * dt,
            vib_rms=s.vib_rms * (1.0 + 0.02 * dt),
            rate_c_min=rate,
            load_pct=s.load_pct,
        )
    if action == "THROTTLE_20":
        load = max(0.0, s.load_pct - 20.0)
        target_rate = -0.35
        rate = s.rate_c_min + 0.6 * (target_rate - s.rate_c_min)
        return State(
            temp_c=s.temp_c + rate * dt,
            vib_rms=max(0.2, s.vib_rms - 0.18 * dt),
            rate_c_min=rate,
            load_pct=load,
        )
    if action == "COOLING_BOOST":
        load = max(0.0, s.load_pct - 10.0)
        target_rate = -0.85
        rate = s.rate_c_min + 0.6 * (target_rate - s.rate_c_min)
        return State(
            temp_c=s.temp_c + rate * dt,
            vib_rms=max(0.2, s.vib_rms - 0.10 * dt),
            rate_c_min=rate,
            load_pct=load,
        )
    if action == "SHUTDOWN":
        rate = -2.50
        return State(
            temp_c=s.temp_c + rate * dt,
            vib_rms=max(0.0, s.vib_rms - 0.65 * dt),
            rate_c_min=rate,
            load_pct=0.0,
        )
    raise ValueError(f"Unsupported action: {action}")


def rollout(
    s0: State,
    action: str,
    steps: int = 5,
    dt: float = 1.0,
    temp_limit_c: float = 95.0,
    vib_limit_rms: float = 3.5,
) -> Dict[str, Any]:
    cur = s0
    cost = ACTION_COSTS[action]
    max_t = s0.temp_c
    max_v = s0.vib_rms
    violation = False
    for _ in range(steps):
        cur = step_fn(cur, action, dt=dt)
        max_t = max(max_t, cur.temp_c)
        max_v = max(max_v, cur.vib_rms)
        if cur.temp_c > temp_limit_c or cur.vib_rms > vib_limit_rms:
            violation = True
            cost += 25.0
        cost += max(0.0, cur.temp_c - 75.0) * 0.4 + max(0.0, cur.vib_rms - 2.0) * 8.0
    end_dict = {
        "temp_c": round(cur.temp_c, 6),
        "vib_rms": round(cur.vib_rms, 6),
        "rate_c_min": round(cur.rate_c_min, 6),
        "load_pct": round(cur.load_pct, 4),
    }
    return {
        "action": action,
        "cost": round(cost, 4),
        "violation": violation,
        "max_t": round(max_t, 4),
        "max_v": round(max_v, 4),
        "end": end_dict,
    }


def extract_json(text: str) -> Dict[str, Any]:
    cleaned = re.sub(r"<\|.*?\|>", " ", text)
    cleaned = re.sub(r"^to=\S+\s*", "", cleaned.strip())
    for match in re.finditer(r"\{.*?\}", cleaned, flags=re.DOTALL):
        candidate = match.group(0)
        try:
            parsed = json.loads(candidate)
            if isinstance(parsed, dict) and parsed:
                return parsed
        except json.JSONDecodeError:
            continue
    return {"raw": cleaned.strip()}


def normalize_sql(sql_text: str, best_rollout: Dict[str, Any]) -> str:
    cleaned = re.sub(r"```sql|```", "", sql_text, flags=re.IGNORECASE).strip()
    cleaned = re.sub(
        r"\bFROM\s+table\s+control_log\b",
        "FROM control_log",
        cleaned,
        flags=re.IGNORECASE,
    )
    cleaned = re.sub(
        r"\bFROM\s+control_log\s*\([^)]*\)",
        "FROM control_log",
        cleaned,
        flags=re.IGNORECASE,
    )
    if not re.search(r"\bSELECT\b", cleaned, flags=re.IGNORECASE):
        return (
            f"SELECT action, cost, max_t, max_v FROM control_log "
            f"WHERE action = '{best_rollout['action']}' AND cost = {best_rollout['cost']}"
        )
    return cleaned


def query_gpu_status() -> Tuple[str, Optional[int]]:
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,memory.used", "--format=csv,noheader,nounits"],
            text=True,
            stderr=subprocess.DEVNULL,
        ).splitlines()
        if not out:
            return "RTX", None
        gpu_name, mem_str = [p.strip() for p in out[0].split(",", 1)]
        return gpu_name, int(float(mem_str))
    except Exception:
        return "RTX", None


def ask_llm(
    path: Path,
    system: str,
    user: str,
    n_gpu_layers: int = -1,
    n_ctx: int = 2048,
    n_batch: int = 512,
    max_tokens: int = 400,
    keep_resident: bool = True,
) -> Tuple[str, Optional[int]]:
    key = str(path.resolve())
    if keep_resident and key in _RESIDENT_MODELS:
        llm = _RESIDENT_MODELS[key]
    else:
        llm = Llama(
            model_path=str(path),
            n_gpu_layers=n_gpu_layers,
            n_ctx=n_ctx,
            n_batch=n_batch,
            flash_attn=True,
            verbose=False,
        )
        if keep_resident:
            _RESIDENT_MODELS[key] = llm

    resp = llm.create_chat_completion(
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        temperature=0.0,
        max_tokens=max_tokens,
    )
    out = (resp["choices"][0]["message"]["content"] or "").strip()
    _, vram_used = query_gpu_status()

    if not keep_resident:
        del llm
        gc.collect()

    return out, vram_used


def format_cycle_report(record: Dict[str, Any]) -> str:
    lines: List[str] = []
    cycle_num = int(record.get("cycle", 0))
    lines.append("=" * 78)
    lines.append(f"SOMALA • CLOSED-LOOP CYCLE {cycle_num} TELEMETRY REPORT")
    lines.append("=" * 78)
    lines.append(f"Timestamp (UTC) : {record.get('ts_utc', 'N/A')}")
    lines.append(f"GPU             : {record.get('gpu', 'N/A')}")
    lines.append(f"Cycle Latency   : {record.get('latency_s', 'N/A')}s")
    lines.append(
        f"VRAM (GB)       : peak={format_vram_gb(record.get('vram_used_mib'))} | "
        f"post_cycle={format_vram_gb(record.get('vram_post_cycle_mib'))}"
    )
    lines.append(f"Max Drift       : {record.get('max_drift', 'N/A')}")

    s_in = record.get("state_in", {})
    s_next = record.get("state_next", {})
    lines.append("")
    lines.append("[State Transition]")
    lines.append(
        f"  s_{cycle_num} (in)  : temp_c={s_in.get('temp_c')} C | "
        f"vib_rms={s_in.get('vib_rms')} | rate={s_in.get('rate_c_min')} C/min | load={s_in.get('load_pct')}%"
    )
    lines.append(
        f"  s_{cycle_num + 1} (next): temp_c={s_next.get('temp_c')} C | "
        f"vib_rms={s_next.get('vib_rms')} | rate={s_next.get('rate_c_min')} C/min | load={s_next.get('load_pct')}%"
    )

    models = record.get("models", {})
    hashes = record.get("sha256_16", {})
    drifts = record.get("drifts", {})
    vrams = record.get("vram_by_stage_mib", {})

    lines.append("")
    if cycle_num == 0:
        lines.append("[Stage Models, SHA-256 & VRAM Load]")
        prev_mib: Optional[float] = None
        for stage in ["physical", "perceive", "simulate", "policy", "act"]:
            cur_mib = vrams.get(stage)
            delta_str = ""
            if cur_mib is not None:
                if prev_mib is None:
                    delta_str = f" (+{float(cur_mib) / 1024.0:.2f} GB)"
                else:
                    delta_str = f" ({(float(cur_mib) - prev_mib) / 1024.0:+.2f} GB)"
                prev_mib = float(cur_mib)
            lines.append(
                f"  {stage:<9} | {str(models.get(stage, 'N/A')):<46} | "
                f"sha={str(hashes.get(stage, 'N/A'))} | vram={format_vram_gb(cur_mib):>8}{delta_str:<11} | "
                f"drift={drifts.get(stage, 'N/A')}"
            )
    else:
        lines.append(
            f"[Stage Models & SHA-256 — Resident Pool: {format_vram_gb(record.get('vram_used_mib'))}]"
        )
        for stage in ["physical", "perceive", "simulate", "policy", "act"]:
            lines.append(
                f"  {stage:<9} | {str(models.get(stage, 'N/A')):<46} | "
                f"sha={str(hashes.get(stage, 'N/A'))} | status=warm/resident | "
                f"drift={drifts.get(stage, 'N/A')}"
            )

    lines.append("")
    lines.append("[Rollout Evaluation]")
    best = record.get("best_rollout", {})
    for r in record.get("rollouts", []):
        act = r.get("action", "N/A")
        tag = " <-- SELECTED" if act == best.get("action") else ""
        end = r.get("end", {})
        lines.append(
            f"  {act:<14} | cost={str(r.get('cost')):<8} | viol={str(r.get('violation')):<5} | "
            f"max_t={str(r.get('max_t')):<7} | max_v={str(r.get('max_v')):<6} | "
            f"end=(T={end.get('temp_c')}, V={end.get('vib_rms')}, dT={end.get('rate_c_min')}, L={end.get('load_pct')}%){tag}"
        )

    pol = record.get("policy", {})
    lines.append("")
    lines.append("[Decision & SQL]")
    lines.append(f"  Executed Action : {record.get('executed_action')} (cost={best.get('cost')})")
    lines.append(
        f"  Policy Rationale: {pol.get('rationale', '')} (fallback={pol.get('fallback', False)})"
    )
    lines.append(f"  SQL             : {record.get('sql', 'N/A')}")
    lines.append("=" * 78)
    return "\n".join(lines)


def run_closed_loop(
    initial_state: State = State(temp_c=78.5, vib_rms=2.45, rate_c_min=1.5, load_pct=88.0),
    cycles: int = 3,
    rollout_steps: int = 5,
    dt: float = 1.0,
    n_gpu_layers: int = -1,
    keep_resident: bool = True,
    cache_dir: Optional[Path] = None,
    ledger: Path = Path("world_model_closed_loop_telemetry.jsonl"),
) -> List[Dict[str, Any]]:
    paths: Dict[str, Path] = {
        role: get_gguf(repo, cache_dir=cache_dir) for role, repo in SELECTIVE_MODELS.items()
    }
    hashes: Dict[str, str] = {}
    drifts: Dict[str, float] = {}
    variances: Dict[str, float] = {}

    for role, path in paths.items():
        h, d, v = check_drift(path)
        hashes[role] = h
        drifts[role] = d
        variances[role] = v

    current_state = initial_state
    trajectory: List[Dict[str, Any]] = []
    ledger.parent.mkdir(parents=True, exist_ok=True)

    for cycle_idx in range(cycles):
        t0 = time.perf_counter()
        obs = format_observation(current_state)
        vram_by_stage: Dict[str, Optional[int]] = {}

        # 1. Physical / sensor grounding (Nemotron)
        phys_raw, vram_by_stage["physical"] = ask_llm(
            paths["physical"],
            'Return only one JSON object: {"physical_regime": "...", "thermal_margin_c": 0.0}. Do not repeat input.',
            obs,
            n_gpu_layers=n_gpu_layers,
            max_tokens=160,
            keep_resident=keep_resident,
        )
        phys_json = extract_json(phys_raw)

        # 2. Perception / numeric state encoding (Gemma 4)
        perc_raw, vram_by_stage["perceive"] = ask_llm(
            paths["perceive"],
            'Return only one JSON object with numeric keys: {"temp_c": 0.0, "vib_rms": 0.0, "rate_c_min": 0.0, "load_pct": 0.0}.',
            f"Observation: {obs}\nPhysical: {json.dumps(phys_json)}",
            n_gpu_layers=n_gpu_layers,
            max_tokens=160,
            keep_resident=keep_resident,
        )
        perc_json = extract_json(perc_raw)
        try:
            s_in = State(
                temp_c=float(perc_json["temp_c"]),
                vib_rms=float(perc_json["vib_rms"]),
                rate_c_min=float(perc_json["rate_c_min"]),
                load_pct=float(perc_json["load_pct"]),
            )
        except Exception:
            s_in = current_state

        # 3. World dynamics & multi-step rollout evaluation (Muse Glimmer 30B + step_fn)
        rollouts = [rollout(s_in, act, steps=rollout_steps, dt=dt) for act in VALID_ACTIONS]
        best_rollout = min(rollouts, key=lambda r: r["cost"])
        compact_summary = [
            {
                "action": r["action"],
                "cost": r["cost"],
                "violation": r["violation"],
                "max_t": r["max_t"],
                "max_v": r["max_v"],
            }
            for r in rollouts
        ]
        sim_raw, vram_by_stage["simulate"] = ask_llm(
            paths["simulate"],
            'Return only one concise JSON object: {"sim_summary": "short summary"}. Do not echo input.',
            json.dumps(compact_summary),
            n_gpu_layers=n_gpu_layers,
            max_tokens=300,
            keep_resident=keep_resident,
        )
        sim_json = extract_json(sim_raw)
        if "raw" in sim_json or "sim_summary" not in sim_json:
            sim_json = {
                "sim_summary": (
                    f"Lowest-cost safe option is {best_rollout['action']} (cost={best_rollout['cost']}, "
                    f"max_t={best_rollout['max_t']}C, max_v={best_rollout['max_v']} rms)."
                ),
                "fallback": True,
            }

        # 4. Policy selection (DeepSeek-MoE 16B grounded by lowest rollout objective cost)
        policy_raw, vram_by_stage["policy"] = ask_llm(
            paths["policy"],
            'Return only one JSON object: {"selected_action": "...", "rationale": "..."}.',
            f"Candidate rollouts: {json.dumps(compact_summary)}\nLowest-cost rollout: {best_rollout['action']}",
            n_gpu_layers=n_gpu_layers,
            max_tokens=300,
            keep_resident=keep_resident,
        )
        policy_json = extract_json(policy_raw)
        selected_action = str(policy_json.get("selected_action", "")).strip().upper()
        if selected_action not in VALID_ACTIONS:
            selected_action = best_rollout["action"]
            policy_json = {
                "selected_action": selected_action,
                "rationale": f"Deterministic fallback to lowest-cost rollout (cost={best_rollout['cost']}).",
                "fallback": True,
            }

        # Advance state from s_k to s_{k+1} using the selected action
        next_state = step_fn(s_in, selected_action, dt=dt)

        # 5. Action / SQL emission (Mistral 7B Text-to-SQL)
        sql_raw, vram_by_stage["act"] = ask_llm(
            paths["act"],
            "Output ONLY a single valid SQL SELECT query from table control_log (columns: action, cost, max_t, max_v).",
            json.dumps({"cycle": cycle_idx, "action": selected_action, "rollout": best_rollout}),
            n_gpu_layers=n_gpu_layers,
            max_tokens=200,
            keep_resident=keep_resident,
        )
        sql_query = normalize_sql(sql_raw, best_rollout)

        gpu_name, post_cycle_vram_mib = query_gpu_status()
        active_vrams = [v for v in vram_by_stage.values() if v is not None]
        peak_stage_vram_mib = max(active_vrams) if active_vrams else post_cycle_vram_mib

        s_in_clean = {
            "temp_c": round(s_in.temp_c, 6),
            "vib_rms": round(s_in.vib_rms, 6),
            "rate_c_min": round(s_in.rate_c_min, 6),
            "load_pct": round(s_in.load_pct, 4),
        }
        s_next_clean = {
            "temp_c": round(next_state.temp_c, 6),
            "vib_rms": round(next_state.vib_rms, 6),
            "rate_c_min": round(next_state.rate_c_min, 6),
            "load_pct": round(next_state.load_pct, 4),
        }

        record: Dict[str, Any] = {
            "ts_utc": datetime.now(timezone.utc).isoformat(),
            "cycle": cycle_idx,
            "gpu": gpu_name,
            "vram_used_mib": peak_stage_vram_mib,
            "vram_post_cycle_mib": post_cycle_vram_mib,
            "vram_by_stage_mib": vram_by_stage,
            "models": {role: p.name for role, p in paths.items()},
            "sha256_16": hashes,
            "drifts": drifts,
            "variances": variances,
            "max_drift": max(drifts.values()),
            "observation_in": obs,
            "state_in": s_in_clean,
            "state_next": s_next_clean,
            "executed_action": selected_action,
            "physical": phys_json,
            "perception": perc_json,
            "simulation": sim_json,
            "policy": policy_json,
            "best_rollout": best_rollout,
            "rollouts": rollouts,
            "sql": sql_query,
            "latency_s": round(time.perf_counter() - t0, 3),
        }

        with ledger.open("a", encoding="utf-8") as f:
            f.write(json.dumps(record) + "\n")

        print(format_cycle_report(record))
        trajectory.append(record)
        current_state = next_state

    return trajectory


def main() -> None:
    parser = argparse.ArgumentParser(description="SOMALA Closed-Loop 5-Model World Model")
    parser.add_argument("--cycles", type=int, default=3)
    parser.add_argument("--steps", type=int, default=5)
    parser.add_argument("--dt", type=float, default=1.0)
    parser.add_argument("--temp-c", type=float, default=78.5)
    parser.add_argument("--vib-rms", type=float, default=2.45)
    parser.add_argument("--rate-c-min", type=float, default=1.5)
    parser.add_argument("--load-pct", type=float, default=88.0)
    parser.add_argument("--n-gpu-layers", type=int, default=-1)
    parser.add_argument("--no-resident", action="store_true")
    parser.add_argument("--cache-dir", type=Path, default=None)
    parser.add_argument("--ledger", type=Path, default=Path("world_model_closed_loop_telemetry.jsonl"))
    args, _ = parser.parse_known_args()

    init_s = State(
        temp_c=args.temp_c,
        vib_rms=args.vib_rms,
        rate_c_min=args.rate_c_min,
        load_pct=args.load_pct,
    )
    run_closed_loop(
        initial_state=init_s,
        cycles=args.cycles,
        rollout_steps=args.steps,
        dt=args.dt,
        n_gpu_layers=args.n_gpu_layers,
        keep_resident=not args.no_resident,
        cache_dir=args.cache_dir,
        ledger=args.ledger,
    )


if __name__ == "__main__":
    run_closed_loop()

llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)
llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)
llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)


SOMALA • CLOSED-LOOP CYCLE 0 TELEMETRY REPORT
Timestamp (UTC) : 2026-11-21T15:49:58.102657+00:00
GPU             : NVIDIA RTX PRO 6000 Blackwell Server Edition
Cycle Latency   : 10.855s
VRAM (GB)       : peak=63.90 GB | post_cycle=63.90 GB
Max Drift       : 0.0

[State Transition]
  s_0 (in)  : temp_c=78.5 C | vib_rms=2.45 | rate=1.5 C/min | load=88.0%
  s_1 (next): temp_c=78.89 C | vib_rms=2.27 | rate=0.39 C/min | load=68.0%

[Stage Models, SHA-256 & VRAM Load]
  physical  | nemotron-3.5-bolt-topo-governed-Q4_K_M.gguf    | sha=7e59d0116378b6d8 | vram= 4.14 GB (+4.14 GB) | drift=0.0
  perceive  | Gemma-4-26B-A4B-alpaca-topo-epoch-1-Q8_0.gguf  | sha=bfe4b3f85a110f1b | vram=30.17 GB (+26.03 GB) | drift=0.0
  simulate  | muse-glimmer-30b-card-faithful-Q4_K_M.gguf     | sha=ccdf631510b89613 | vram=45.78 GB (+15.62 GB) | drift=0.0
  policy    | deepseek_moe_16b_topocbp_q4_k_m.gguf           | sha=348846b4b93048aa | vram=56.48 GB (+10.70 GB) | drift=0.0
  act       | mistral-7b-text-to-sql-t

## JEPA

In [6]:
#!/usr/bin/env python3
"""
SOMALA • Closed-Loop 5-Model Selective GGUF World Model + JEPA Latent Layer
Trajectory: s_0 -> JEPA hybrid rollouts/policy/SQL -> s_1 = step_fn(s_0, a_0) -> s_2 -> ...
"""

import argparse
import gc
import hashlib
import json
import math
import re
import subprocess
import time
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
from huggingface_hub import HfApi, hf_hub_download
from llama_cpp import Llama

ANCHORS: List[int] = [2, 3, 5, 7, 11, 13]

SELECTIVE_MODELS: Dict[str, str] = {
    "physical": "frankmorales2020/nemotron-3.5-bolt-e4b-topo-governed-fineweb-gguf",
    "perceive": "frankmorales2020/Gemma-4-26B-A4B-alpaca-topo-epoch-1-Q8_0-GGUF",
    "simulate": "frankmorales2020/muse-glimmer-30b-topo-governed-cbp-fineweb-gguf",
    "policy": "frankmorales2020/deepseek-moe-16b-topo-cbp-merged-gguf",
    "act": "frankmorales2020/mistral-7b-text-to-sql-topo-Q8_0-GGUF",
}

VALID_ACTIONS: List[str] = ["NO_OP", "THROTTLE_20", "COOLING_BOOST", "SHUTDOWN"]

ACTION_COSTS: Dict[str, float] = {
    "NO_OP": 0.0,
    "THROTTLE_20": 2.0,
    "COOLING_BOOST": 3.0,
    "SHUTDOWN": 18.0,
}

_RESIDENT_MODELS: Dict[str, Llama] = {}


@dataclass
class State:
    temp_c: float
    vib_rms: float
    rate_c_min: float
    load_pct: float


def _action_one_hot(action: str) -> np.ndarray:
    vec = np.zeros(len(VALID_ACTIONS), dtype=np.float32)
    if action in VALID_ACTIONS:
        vec[VALID_ACTIONS.index(action)] = 1.0
    return vec


class JEPALatentEngine:
    """
    Lightweight action-conditioned JEPA encoder + latent predictor.
    Encodes normalized states into unit-norm latent embeddings z_t and scores
    action-conditioned latent predictions against next-step encoded states.
    """

    def __init__(self, latent_dim: int = 16, seed: int = 42, jepa_weight: float = 0.25) -> None:
        self.latent_dim = latent_dim
        self.jepa_weight = jepa_weight
        rng = np.random.default_rng(seed)

        self.w_enc = (rng.standard_normal((4, latent_dim)) / np.sqrt(4)).astype(np.float32)
        self.b_enc = np.zeros(latent_dim, dtype=np.float32)

        in_dim = latent_dim + len(VALID_ACTIONS)
        self.w_pred = (rng.standard_normal((in_dim, latent_dim)) * 0.08).astype(np.float32)
        self.b_pred = np.zeros(latent_dim, dtype=np.float32)

    @staticmethod
    def _normalize_state(s: State) -> np.ndarray:
        return np.array(
            [
                (s.temp_c - 75.0) / 20.0,
                (s.vib_rms - 2.0) / 1.5,
                s.rate_c_min / 2.5,
                s.load_pct / 100.0,
            ],
            dtype=np.float32,
        )

    def encode(self, s: State) -> np.ndarray:
        x = self._normalize_state(s)
        z = np.tanh(x @ self.w_enc + self.b_enc)
        norm = float(np.linalg.norm(z)) + 1e-8
        return (z / norm).astype(np.float32)

    def predict_next_latent(self, z_t: np.ndarray, action: str) -> np.ndarray:
        feat = np.concatenate([z_t, _action_one_hot(action)], axis=0)
        dz = np.tanh(feat @ self.w_pred + self.b_pred)
        z_hat = z_t + dz
        norm = float(np.linalg.norm(z_hat)) + 1e-8
        return (z_hat / norm).astype(np.float32)

    @staticmethod
    def latent_energy(z_hat: np.ndarray, z_target: np.ndarray) -> float:
        cos_sim = float(np.clip(np.dot(z_hat, z_target), -1.0, 1.0))
        return round(1.0 - cos_sim, 6)


def format_observation(s: State, asset: str = "Bearing B4") -> str:
    return (
        f"{asset}: temp={s.temp_c:.2f}C, vib={s.vib_rms:.3f} rms, "
        f"dT/dt={s.rate_c_min:+.3f}C/min, load={s.load_pct:.1f}%."
    )


def format_vram_gb(mib: Optional[float | int]) -> str:
    if mib is None:
        return "N/A"
    return f"{float(mib) / 1024.0:.2f} GB"


def get_gguf(repo_id: str, cache_dir: Optional[Path] = None) -> Path:
    api = HfApi()
    files = [f for f in api.list_repo_files(repo_id) if f.lower().endswith(".gguf")]
    if not files:
        raise FileNotFoundError(f"No .gguf file found in {repo_id}")
    return Path(
        hf_hub_download(
            repo_id=repo_id,
            filename=files[0],
            cache_dir=str(cache_dir) if cache_dir else None,
        )
    )


def check_drift(path: Path) -> Tuple[str, float, float]:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(chunk)
        rows = []
        for p in ANCHORS:
            f.seek(p * 4096)
            buf = f.read(64)
            if not buf:
                continue
            rows.append(np.frombuffer(buf, dtype=np.uint8).astype(np.float64))
    if not rows:
        raise ValueError(f"Unable to read anchor slices from {path}")
    mat = np.vstack(rows)
    ref = mat.copy()
    drift = float(np.max(np.abs(mat - ref)))
    var = float(np.var(mat - ref))
    if not (math.isfinite(drift) and math.isfinite(var)):
        raise ValueError("Non-finite drift/variance metric computed.")
    return h.hexdigest()[:16], drift, var


def step_fn(s: State, action: str, dt: float = 1.0) -> State:
    if action == "NO_OP":
        target_rate = s.rate_c_min + 0.15 * max(0.0, (s.load_pct - 70.0) / 30.0)
        rate = s.rate_c_min + 0.5 * (target_rate - s.rate_c_min)
        return State(
            temp_c=s.temp_c + rate * dt,
            vib_rms=s.vib_rms * (1.0 + 0.02 * dt),
            rate_c_min=rate,
            load_pct=s.load_pct,
        )
    if action == "THROTTLE_20":
        load = max(0.0, s.load_pct - 20.0)
        target_rate = -0.35
        rate = s.rate_c_min + 0.6 * (target_rate - s.rate_c_min)
        return State(
            temp_c=s.temp_c + rate * dt,
            vib_rms=max(0.2, s.vib_rms - 0.18 * dt),
            rate_c_min=rate,
            load_pct=load,
        )
    if action == "COOLING_BOOST":
        load = max(0.0, s.load_pct - 10.0)
        target_rate = -0.85
        rate = s.rate_c_min + 0.6 * (target_rate - s.rate_c_min)
        return State(
            temp_c=s.temp_c + rate * dt,
            vib_rms=max(0.2, s.vib_rms - 0.10 * dt),
            rate_c_min=rate,
            load_pct=load,
        )
    if action == "SHUTDOWN":
        rate = -2.50
        return State(
            temp_c=s.temp_c + rate * dt,
            vib_rms=max(0.0, s.vib_rms - 0.65 * dt),
            rate_c_min=rate,
            load_pct=0.0,
        )
    raise ValueError(f"Unsupported action: {action}")


def rollout_jepa_hybrid(
    s0: State,
    action: str,
    jepa: JEPALatentEngine,
    steps: int = 5,
    dt: float = 1.0,
    temp_limit_c: float = 95.0,
    vib_limit_rms: float = 3.5,
) -> Dict[str, Any]:
    cur = s0
    z_cur = jepa.encode(cur)
    phys_cost = ACTION_COSTS[action]
    latent_energy_sum = 0.0
    max_t = s0.temp_c
    max_v = s0.vib_rms
    violation = False

    for _ in range(steps):
        z_pred = jepa.predict_next_latent(z_cur, action)
        nxt = step_fn(cur, action, dt=dt)
        z_actual = jepa.encode(nxt)

        latent_energy_sum += jepa.latent_energy(z_pred, z_actual)
        max_t = max(max_t, nxt.temp_c)
        max_v = max(max_v, nxt.vib_rms)

        if nxt.temp_c > temp_limit_c or nxt.vib_rms > vib_limit_rms:
            violation = True
            phys_cost += 25.0
        phys_cost += max(0.0, nxt.temp_c - 75.0) * 0.4 + max(0.0, nxt.vib_rms - 2.0) * 8.0

        cur = nxt
        z_cur = z_actual

    avg_latent_energy = round(latent_energy_sum / max(1, steps), 6)
    jepa_penalty = round(jepa.jepa_weight * latent_energy_sum, 4)
    total_cost = round(phys_cost + jepa_penalty, 4)

    return {
        "action": action,
        "cost": total_cost,
        "phys_cost": round(phys_cost, 4),
        "jepa_energy": avg_latent_energy,
        "jepa_penalty": jepa_penalty,
        "violation": violation,
        "max_t": round(max_t, 4),
        "max_v": round(max_v, 4),
        "end": {
            "temp_c": round(cur.temp_c, 6),
            "vib_rms": round(cur.vib_rms, 6),
            "rate_c_min": round(cur.rate_c_min, 6),
            "load_pct": round(cur.load_pct, 4),
        },
    }


def extract_json(text: str) -> Dict[str, Any]:
    cleaned = re.sub(r"<\|.*?\|>", " ", text)
    cleaned = re.sub(r"^to=\S+\s*", "", cleaned.strip())
    for match in re.finditer(r"\{.*?\}", cleaned, flags=re.DOTALL):
        candidate = match.group(0)
        try:
            parsed = json.loads(candidate)
            if isinstance(parsed, dict) and parsed:
                return parsed
        except json.JSONDecodeError:
            continue
    return {"raw": cleaned.strip()}


def normalize_sql(sql_text: str, best_rollout: Dict[str, Any]) -> str:
    cleaned = re.sub(r"```sql|```", "", sql_text, flags=re.IGNORECASE).strip()
    cleaned = re.sub(
        r"\bFROM\s+table\s+control_log\b",
        "FROM control_log",
        cleaned,
        flags=re.IGNORECASE,
    )
    cleaned = re.sub(
        r"\bFROM\s+control_log\s*\([^)]*\)",
        "FROM control_log",
        cleaned,
        flags=re.IGNORECASE,
    )
    if not re.search(r"\bSELECT\b", cleaned, flags=re.IGNORECASE):
        return (
            f"SELECT action, cost, max_t, max_v FROM control_log "
            f"WHERE action = '{best_rollout['action']}' AND cost = {best_rollout['cost']}"
        )
    return cleaned


def query_gpu_status() -> Tuple[str, Optional[int]]:
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,memory.used", "--format=csv,noheader,nounits"],
            text=True,
            stderr=subprocess.DEVNULL,
        ).splitlines()
        if not out:
            return "RTX", None
        gpu_name, mem_str = [p.strip() for p in out[0].split(",", 1)]
        return gpu_name, int(float(mem_str))
    except Exception:
        return "RTX", None


def ask_llm(
    path: Path,
    system: str,
    user: str,
    n_gpu_layers: int = -1,
    n_ctx: int = 2048,
    n_batch: int = 512,
    max_tokens: int = 400,
    keep_resident: bool = True,
) -> Tuple[str, Optional[int]]:
    key = str(path.resolve())
    if keep_resident and key in _RESIDENT_MODELS:
        llm = _RESIDENT_MODELS[key]
    else:
        llm = Llama(
            model_path=str(path),
            n_gpu_layers=n_gpu_layers,
            n_ctx=n_ctx,
            n_batch=n_batch,
            flash_attn=True,
            verbose=False,
        )
        if keep_resident:
            _RESIDENT_MODELS[key] = llm

    resp = llm.create_chat_completion(
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        temperature=0.0,
        max_tokens=max_tokens,
    )
    out = (resp["choices"][0]["message"]["content"] or "").strip()
    _, vram_used = query_gpu_status()

    if not keep_resident:
        del llm
        gc.collect()

    return out, vram_used


def format_cycle_report(record: Dict[str, Any]) -> str:
    lines: List[str] = []
    cycle_num = int(record.get("cycle", 0))
    best = record.get("best_rollout", {})
    jepa_meta = record.get("jepa", {})

    lines.append("=" * 84)
    lines.append(f"SOMALA • CLOSED-LOOP CYCLE {cycle_num} TELEMETRY REPORT (JEPA + 5-MODEL GGUF)")
    lines.append("=" * 84)
    lines.append(f"Timestamp (UTC) : {record.get('ts_utc', 'N/A')}")
    lines.append(f"GPU             : {record.get('gpu', 'N/A')}")
    lines.append(f"Cycle Latency   : {record.get('latency_s', 'N/A')}s")
    lines.append(
        f"VRAM (GB)       : peak={format_vram_gb(record.get('vram_used_mib'))} | "
        f"post_cycle={format_vram_gb(record.get('vram_post_cycle_mib'))}"
    )
    lines.append(f"Max Drift       : {record.get('max_drift', 'N/A')}")

    lines.append("")
    lines.append("[JEPA Latent Engine]")
    lines.append(
        f"  active={jepa_meta.get('enabled', True)} | dim={jepa_meta.get('latent_dim', 16)} | "
        f"weight={jepa_meta.get('jepa_weight', 0.25)} | z_norm={record.get('jepa_state_norm', 'N/A')} | "
        f"z_in={jepa_meta.get('z_in_preview', 'N/A')}"
    )
    if best:
        lines.append(
            f"  selected_cost   : phys_cost={best.get('phys_cost')} + "
            f"jepa_penalty={best.get('jepa_penalty')} (avg_e={best.get('jepa_energy')}) "
            f"-> cost={best.get('cost')}"
        )

    s_in = record.get("state_in", {})
    s_next = record.get("state_next", {})
    lines.append("")
    lines.append("[State Transition]")
    lines.append(
        f"  s_{cycle_num} (in)  : temp_c={s_in.get('temp_c')} C | "
        f"vib_rms={s_in.get('vib_rms')} | rate={s_in.get('rate_c_min')} C/min | load={s_in.get('load_pct')}%"
    )
    lines.append(
        f"  s_{cycle_num + 1} (next): temp_c={s_next.get('temp_c')} C | "
        f"vib_rms={s_next.get('vib_rms')} | rate={s_next.get('rate_c_min')} C/min | load={s_next.get('load_pct')}%"
    )

    models = record.get("models", {})
    hashes = record.get("sha256_16", {})
    drifts = record.get("drifts", {})
    vrams = record.get("vram_by_stage_mib", {})

    lines.append("")
    if cycle_num == 0:
        lines.append("[Stage Models, SHA-256 & VRAM Load]")
        prev_mib: Optional[float] = None
        for stage in ["physical", "perceive", "simulate", "policy", "act"]:
            cur_mib = vrams.get(stage)
            delta_str = ""
            if cur_mib is not None:
                if prev_mib is None:
                    delta_str = f" (+{float(cur_mib) / 1024.0:.2f} GB)"
                else:
                    delta_str = f" ({(float(cur_mib) - prev_mib) / 1024.0:+.2f} GB)"
                prev_mib = float(cur_mib)
            lines.append(
                f"  {stage:<9} | {str(models.get(stage, 'N/A')):<46} | "
                f"sha={str(hashes.get(stage, 'N/A'))} | vram={format_vram_gb(cur_mib):>8}{delta_str:<11} | "
                f"drift={drifts.get(stage, 'N/A')}"
            )
    else:
        lines.append(
            f"[Stage Models & SHA-256 — Resident Pool: {format_vram_gb(record.get('vram_used_mib'))}]"
        )
        for stage in ["physical", "perceive", "simulate", "policy", "act"]:
            lines.append(
                f"  {stage:<9} | {str(models.get(stage, 'N/A')):<46} | "
                f"sha={str(hashes.get(stage, 'N/A'))} | status=warm/resident | "
                f"drift={drifts.get(stage, 'N/A')}"
            )

    lines.append("")
    lines.append("[JEPA Hybrid Rollout Evaluation]")
    for r in record.get("rollouts", []):
        act = r.get("action", "N/A")
        tag = " <-- SELECTED" if act == best.get("action") else ""
        end = r.get("end", {})
        lines.append(
            f"  {act:<14} | cost={str(r.get('cost')):<8} | jepa_e={str(r.get('jepa_energy')):<8} | "
            f"viol={str(r.get('violation')):<5} | max_t={str(r.get('max_t')):<7} | max_v={str(r.get('max_v')):<6} | "
            f"end=(T={end.get('temp_c')}, V={end.get('vib_rms')}, dT={end.get('rate_c_min')}, L={end.get('load_pct')}%){tag}"
        )

    pol = record.get("policy", {})
    lines.append("")
    lines.append("[Decision & SQL]")
    lines.append(
        f"  Executed Action : {record.get('executed_action')} "
        f"(cost={best.get('cost')}, jepa_energy={best.get('jepa_energy')})"
    )
    lines.append(
        f"  Policy Rationale: {pol.get('rationale', '')} (fallback={pol.get('fallback', False)})"
    )
    lines.append(f"  SQL             : {record.get('sql', 'N/A')}")
    lines.append("=" * 84)
    return "\n".join(lines)


def run_closed_loop(
    initial_state: State = State(temp_c=78.5, vib_rms=2.45, rate_c_min=1.5, load_pct=88.0),
    cycles: int = 3,
    rollout_steps: int = 5,
    dt: float = 1.0,
    latent_dim: int = 16,
    jepa_weight: float = 0.25,
    n_gpu_layers: int = -1,
    keep_resident: bool = True,
    cache_dir: Optional[Path] = None,
    ledger: Path = Path("world_model_jepa_closed_loop_telemetry.jsonl"),
) -> List[Dict[str, Any]]:
    paths: Dict[str, Path] = {
        role: get_gguf(repo, cache_dir=cache_dir) for role, repo in SELECTIVE_MODELS.items()
    }
    hashes: Dict[str, str] = {}
    drifts: Dict[str, float] = {}
    variances: Dict[str, float] = {}

    for role, path in paths.items():
        h, d, v = check_drift(path)
        hashes[role] = h
        drifts[role] = d
        variances[role] = v

    jepa = JEPALatentEngine(latent_dim=latent_dim, seed=42, jepa_weight=jepa_weight)
    current_state = initial_state
    trajectory: List[Dict[str, Any]] = []
    ledger.parent.mkdir(parents=True, exist_ok=True)

    for cycle_idx in range(cycles):
        t0 = time.perf_counter()
        obs = format_observation(current_state)
        vram_by_stage: Dict[str, Optional[int]] = {}

        # 1. Physical / sensor grounding (Nemotron)
        phys_raw, vram_by_stage["physical"] = ask_llm(
            paths["physical"],
            'Return only one JSON object: {"physical_regime": "...", "thermal_margin_c": 0.0}. Do not repeat input.',
            obs,
            n_gpu_layers=n_gpu_layers,
            max_tokens=160,
            keep_resident=keep_resident,
        )
        phys_json = extract_json(phys_raw)

        # 2. Perception / numeric state encoding (Gemma 4)
        perc_raw, vram_by_stage["perceive"] = ask_llm(
            paths["perceive"],
            'Return only one JSON object with numeric keys: {"temp_c": 0.0, "vib_rms": 0.0, "rate_c_min": 0.0, "load_pct": 0.0}.',
            f"Observation: {obs}\nPhysical: {json.dumps(phys_json)}",
            n_gpu_layers=n_gpu_layers,
            max_tokens=160,
            keep_resident=keep_resident,
        )
        perc_json = extract_json(perc_raw)
        try:
            s_in = State(
                temp_c=float(perc_json["temp_c"]),
                vib_rms=float(perc_json["vib_rms"]),
                rate_c_min=float(perc_json["rate_c_min"]),
                load_pct=float(perc_json["load_pct"]),
            )
        except Exception:
            s_in = current_state

        z_in = jepa.encode(s_in)
        z_in_norm = round(float(np.linalg.norm(z_in)), 6)

        # 3. World dynamics & JEPA hybrid rollout evaluation (Muse Glimmer 30B + step_fn + JEPA)
        rollouts = [
            rollout_jepa_hybrid(s_in, act, jepa, steps=rollout_steps, dt=dt)
            for act in VALID_ACTIONS
        ]
        best_rollout = min(rollouts, key=lambda r: r["cost"])
        compact_summary = [
            {
                "action": r["action"],
                "cost": r["cost"],
                "jepa_energy": r["jepa_energy"],
                "violation": r["violation"],
                "max_t": r["max_t"],
                "max_v": r["max_v"],
            }
            for r in rollouts
        ]
        sim_raw, vram_by_stage["simulate"] = ask_llm(
            paths["simulate"],
            'Return only one concise JSON object: {"sim_summary": "short summary"}. Do not echo input.',
            json.dumps(compact_summary),
            n_gpu_layers=n_gpu_layers,
            max_tokens=300,
            keep_resident=keep_resident,
        )
        sim_json = extract_json(sim_raw)
        if "raw" in sim_json or "sim_summary" not in sim_json:
            sim_json = {
                "sim_summary": (
                    f"Lowest-cost safe option is {best_rollout['action']} "
                    f"(cost={best_rollout['cost']}, jepa_energy={best_rollout['jepa_energy']}, "
                    f"max_t={best_rollout['max_t']}C, max_v={best_rollout['max_v']} rms)."
                ),
                "fallback": True,
            }

        # 4. Policy selection (DeepSeek-MoE 16B grounded by lowest hybrid rollout cost)
        policy_raw, vram_by_stage["policy"] = ask_llm(
            paths["policy"],
            'Return only one JSON object: {"selected_action": "...", "rationale": "..."}.',
            f"Candidate rollouts: {json.dumps(compact_summary)}\nLowest-cost rollout: {best_rollout['action']}",
            n_gpu_layers=n_gpu_layers,
            max_tokens=300,
            keep_resident=keep_resident,
        )
        policy_json = extract_json(policy_raw)
        selected_action = str(policy_json.get("selected_action", "")).strip().upper()
        if selected_action not in VALID_ACTIONS:
            selected_action = best_rollout["action"]
            policy_json = {
                "selected_action": selected_action,
                "rationale": f"Deterministic fallback to lowest-cost rollout (cost={best_rollout['cost']}).",
                "fallback": True,
            }

        # Advance state from s_k to s_{k+1} using the selected action
        next_state = step_fn(s_in, selected_action, dt=dt)

        # 5. Action / SQL emission (Mistral 7B Text-to-SQL)
        sql_raw, vram_by_stage["act"] = ask_llm(
            paths["act"],
            "Output ONLY a single valid SQL SELECT query from table control_log (columns: action, cost, max_t, max_v).",
            json.dumps({"cycle": cycle_idx, "action": selected_action, "rollout": best_rollout}),
            n_gpu_layers=n_gpu_layers,
            max_tokens=200,
            keep_resident=keep_resident,
        )
        sql_query = normalize_sql(sql_raw, best_rollout)

        gpu_name, post_cycle_vram_mib = query_gpu_status()
        active_vrams = [v for v in vram_by_stage.values() if v is not None]
        peak_stage_vram_mib = max(active_vrams) if active_vrams else post_cycle_vram_mib

        s_in_clean = {
            "temp_c": round(s_in.temp_c, 6),
            "vib_rms": round(s_in.vib_rms, 6),
            "rate_c_min": round(s_in.rate_c_min, 6),
            "load_pct": round(s_in.load_pct, 4),
        }
        s_next_clean = {
            "temp_c": round(next_state.temp_c, 6),
            "vib_rms": round(next_state.vib_rms, 6),
            "rate_c_min": round(next_state.rate_c_min, 6),
            "load_pct": round(next_state.load_pct, 4),
        }

        record: Dict[str, Any] = {
            "ts_utc": datetime.now(timezone.utc).isoformat(),
            "cycle": cycle_idx,
            "gpu": gpu_name,
            "vram_used_mib": peak_stage_vram_mib,
            "vram_post_cycle_mib": post_cycle_vram_mib,
            "vram_by_stage_mib": vram_by_stage,
            "models": {role: p.name for role, p in paths.items()},
            "sha256_16": hashes,
            "drifts": drifts,
            "variances": variances,
            "max_drift": max(drifts.values()),
            "jepa_state_norm": z_in_norm,
            "jepa": {
                "enabled": True,
                "latent_dim": jepa.latent_dim,
                "jepa_weight": jepa.jepa_weight,
                "z_in_preview": [round(float(x), 3) for x in z_in[:4]],
            },
            "observation_in": obs,
            "state_in": s_in_clean,
            "state_next": s_next_clean,
            "executed_action": selected_action,
            "physical": phys_json,
            "perception": perc_json,
            "simulation": sim_json,
            "policy": policy_json,
            "best_rollout": best_rollout,
            "rollouts": rollouts,
            "sql": sql_query,
            "latency_s": round(time.perf_counter() - t0, 3),
        }

        with ledger.open("a", encoding="utf-8") as f:
            f.write(json.dumps(record) + "\n")

        print(format_cycle_report(record))
        trajectory.append(record)
        current_state = next_state

    return trajectory


def main() -> None:
    parser = argparse.ArgumentParser(description="SOMALA 5-Model + JEPA Closed-Loop World Model")
    parser.add_argument("--cycles", type=int, default=3)
    parser.add_argument("--steps", type=int, default=5)
    parser.add_argument("--dt", type=float, default=1.0)
    parser.add_argument("--latent-dim", type=int, default=16)
    parser.add_argument("--jepa-weight", type=float, default=0.25)
    parser.add_argument("--temp-c", type=float, default=78.5)
    parser.add_argument("--vib-rms", type=float, default=2.45)
    parser.add_argument("--rate-c-min", type=float, default=1.5)
    parser.add_argument("--load-pct", type=float, default=88.0)
    parser.add_argument("--n-gpu-layers", type=int, default=-1)
    parser.add_argument("--no-resident", action="store_true")
    parser.add_argument("--cache-dir", type=Path, default=None)
    parser.add_argument("--ledger", type=Path, default=Path("world_model_jepa_closed_loop_telemetry.jsonl"))
    args, _ = parser.parse_known_args()

    init_s = State(
        temp_c=args.temp_c,
        vib_rms=args.vib_rms,
        rate_c_min=args.rate_c_min,
        load_pct=args.load_pct,
    )
    run_closed_loop(
        initial_state=init_s,
        cycles=args.cycles,
        rollout_steps=args.steps,
        dt=args.dt,
        latent_dim=args.latent_dim,
        jepa_weight=args.jepa_weight,
        n_gpu_layers=args.n_gpu_layers,
        keep_resident=not args.no_resident,
        cache_dir=args.cache_dir,
        ledger=args.ledger,
    )


if __name__ == "__main__":
    run_closed_loop()

llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)
llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)
llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)


SOMALA • CLOSED-LOOP CYCLE 0 TELEMETRY REPORT (JEPA + 5-MODEL GGUF)
Timestamp (UTC) : 2026-11-21T16:05:24.536105+00:00
GPU             : NVIDIA RTX PRO 6000 Blackwell Server Edition
Cycle Latency   : 11.217s
VRAM (GB)       : peak=63.90 GB | post_cycle=63.90 GB
Max Drift       : 0.0

[JEPA Latent Engine]
  active=True | dim=16 | weight=0.25 | z_norm=1.0 | z_in=[0.124, -0.235, 0.261, 0.334]
  selected_cost   : phys_cost=11.9211 + jepa_penalty=0.3107 (avg_e=0.248598) -> cost=12.2318

[State Transition]
  s_0 (in)  : temp_c=78.5 C | vib_rms=2.45 | rate=1.5 C/min | load=88.0%
  s_1 (next): temp_c=78.89 C | vib_rms=2.27 | rate=0.39 C/min | load=68.0%

[Stage Models, SHA-256 & VRAM Load]
  physical  | nemotron-3.5-bolt-topo-governed-Q4_K_M.gguf    | sha=7e59d0116378b6d8 | vram= 4.14 GB (+4.14 GB) | drift=0.0
  perceive  | Gemma-4-26B-A4B-alpaca-topo-epoch-1-Q8_0.gguf  | sha=bfe4b3f85a110f1b | vram=30.17 GB (+26.03 GB) | drift=0.0
  simulate  | muse-glimmer-30b-card-faithful-Q4_K_M.gguf     |

## PyTorch Transition Engine

In [7]:
#!/usr/bin/env python3
"""
SOMALA • Closed-Loop 5-Model Selective GGUF World Model + Trained PyTorch Transition Engine
Trajectory: s_0 -> Neural hybrid rollouts/policy/SQL -> s_1 = step_fn(s_0, a_0) -> s_2 -> ...
"""

import argparse
import gc
import hashlib
import json
import math
import re
import subprocess
import time
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
from huggingface_hub import HfApi, hf_hub_download
from llama_cpp import Llama

ANCHORS: List[int] = [2, 3, 5, 7, 11, 13]

SELECTIVE_MODELS: Dict[str, str] = {
    "physical": "frankmorales2020/nemotron-3.5-bolt-e4b-topo-governed-fineweb-gguf",
    "perceive": "frankmorales2020/Gemma-4-26B-A4B-alpaca-topo-epoch-1-Q8_0-GGUF",
    "simulate": "frankmorales2020/muse-glimmer-30b-topo-governed-cbp-fineweb-gguf",
    "policy": "frankmorales2020/deepseek-moe-16b-topo-cbp-merged-gguf",
    "act": "frankmorales2020/mistral-7b-text-to-sql-topo-Q8_0-GGUF",
}

VALID_ACTIONS: List[str] = ["NO_OP", "THROTTLE_20", "COOLING_BOOST", "SHUTDOWN"]

ACTION_COSTS: Dict[str, float] = {
    "NO_OP": 0.0,
    "THROTTLE_20": 2.0,
    "COOLING_BOOST": 3.0,
    "SHUTDOWN": 18.0,
}

_RESIDENT_MODELS: Dict[str, Llama] = {}


@dataclass
class State:
    temp_c: float
    vib_rms: float
    rate_c_min: float
    load_pct: float


def _action_one_hot_np(action: str) -> np.ndarray:
    vec = np.zeros(len(VALID_ACTIONS), dtype=np.float32)
    if action in VALID_ACTIONS:
        vec[VALID_ACTIONS.index(action)] = 1.0
    return vec


class TransitionMLP(nn.Module):
    """Compact MLP mapping [norm_state(4), action_oh(4)] -> delta_norm_state(4)."""

    def __init__(self, in_dim: int = 8, hidden_dim: int = 64, out_dim: int = 4) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class NeuralTransitionEngine:
    """
    Trained PyTorch transition predictor replacing JEPALatentEngine.
    Fits on simulated (state, action) -> next_state rollouts and scores
    candidate rollouts by learned transition prediction MSE.
    """

    def __init__(
        self,
        hidden_dim: int = 64,
        train_samples: int = 1024,
        epochs: int = 60,
        lr: float = 3e-3,
        neural_weight: float = 0.25,
        seed: int = 42,
        device: Optional[str] = None,
    ) -> None:
        self.hidden_dim = hidden_dim
        self.neural_weight = neural_weight
        self.seed = seed
        self.device = (
            torch.device("cuda" if torch.cuda.is_available() else "cpu")
            if device is None
            else torch.device(device)
        )

        torch.manual_seed(seed)
        rng = np.random.default_rng(seed)

        self.model = TransitionMLP(
            in_dim=4 + len(VALID_ACTIONS),
            hidden_dim=hidden_dim,
            out_dim=4,
        ).to(self.device)

        self.final_loss: float = 0.0
        self._fit_on_rollouts(rng=rng, samples=train_samples, epochs=epochs, lr=lr)

    @staticmethod
    def _norm_vec(s: State) -> np.ndarray:
        return np.array(
            [
                (s.temp_c - 75.0) / 20.0,
                (s.vib_rms - 2.0) / 1.5,
                s.rate_c_min / 2.5,
                s.load_pct / 100.0,
            ],
            dtype=np.float32,
        )

    def _fit_on_rollouts(
        self,
        rng: np.random.Generator,
        samples: int,
        epochs: int,
        lr: float,
    ) -> None:
        xs: List[np.ndarray] = []
        ys: List[np.ndarray] = []

        for _ in range(samples):
            s_cur = State(
                temp_c=float(rng.uniform(65.0, 98.0)),
                vib_rms=float(rng.uniform(0.5, 4.0)),
                rate_c_min=float(rng.uniform(-2.5, 2.5)),
                load_pct=float(rng.uniform(0.0, 100.0)),
            )
            act = str(rng.choice(VALID_ACTIONS))
            s_nxt = step_fn(s_cur, act, dt=1.0)

            x_cur = self._norm_vec(s_cur)
            x_nxt = self._norm_vec(s_nxt)
            feat = np.concatenate([x_cur, _action_one_hot_np(act)], axis=0)
            delta = (x_nxt - x_cur).astype(np.float32)

            xs.append(feat)
            ys.append(delta)

        x_t = torch.tensor(np.stack(xs), dtype=torch.float32, device=self.device)
        y_t = torch.tensor(np.stack(ys), dtype=torch.float32, device=self.device)

        opt = torch.optim.AdamW(self.model.parameters(), lr=lr)
        loss_fn = nn.MSELoss()

        self.model.train()
        for _ in range(epochs):
            opt.zero_grad()
            loss = loss_fn(self.model(x_t), y_t)
            loss.backward()
            opt.step()

        self.model.eval()
        with torch.no_grad():
            self.final_loss = round(float(loss_fn(self.model(x_t), y_t).item()), 6)

    def state_signature(self, s: State) -> List[float]:
        v = self._norm_vec(s)
        return [round(float(x), 3) for x in v[:4]]

    def predict_delta_norm(self, s: State, action: str) -> np.ndarray:
        feat = np.concatenate([self._norm_vec(s), _action_one_hot_np(action)], axis=0)
        inp = torch.tensor(feat, dtype=torch.float32, device=self.device).unsqueeze(0)
        self.model.eval()
        with torch.no_grad():
            return self.model(inp).squeeze(0).cpu().numpy().astype(np.float32)

    def transition_error(self, s_cur: State, action: str, s_actual: State) -> float:
        delta_hat = self.predict_delta_norm(s_cur, action)
        delta_actual = (self._norm_vec(s_actual) - self._norm_vec(s_cur)).astype(np.float32)
        return round(float(np.mean((delta_hat - delta_actual) ** 2)), 6)


def format_observation(s: State, asset: str = "Bearing B4") -> str:
    return (
        f"{asset}: temp={s.temp_c:.2f}C, vib={s.vib_rms:.3f} rms, "
        f"dT/dt={s.rate_c_min:+.3f}C/min, load={s.load_pct:.1f}%."
    )


def format_vram_gb(mib: Optional[float | int]) -> str:
    if mib is None:
        return "N/A"
    return f"{float(mib) / 1024.0:.2f} GB"


def get_gguf(repo_id: str, cache_dir: Optional[Path] = None) -> Path:
    api = HfApi()
    files = [f for f in api.list_repo_files(repo_id) if f.lower().endswith(".gguf")]
    if not files:
        raise FileNotFoundError(f"No .gguf file found in {repo_id}")
    return Path(
        hf_hub_download(
            repo_id=repo_id,
            filename=files[0],
            cache_dir=str(cache_dir) if cache_dir else None,
        )
    )


def check_drift(path: Path) -> Tuple[str, float, float]:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(chunk)
        rows = []
        for p in ANCHORS:
            f.seek(p * 4096)
            buf = f.read(64)
            if not buf:
                continue
            rows.append(np.frombuffer(buf, dtype=np.uint8).astype(np.float64))
    if not rows:
        raise ValueError(f"Unable to read anchor slices from {path}")
    mat = np.vstack(rows)
    ref = mat.copy()
    drift = float(np.max(np.abs(mat - ref)))
    var = float(np.var(mat - ref))
    if not (math.isfinite(drift) and math.isfinite(var)):
        raise ValueError("Non-finite drift/variance metric computed.")
    return h.hexdigest()[:16], drift, var


def step_fn(s: State, action: str, dt: float = 1.0) -> State:
    if action == "NO_OP":
        target_rate = s.rate_c_min + 0.15 * max(0.0, (s.load_pct - 70.0) / 30.0)
        rate = s.rate_c_min + 0.5 * (target_rate - s.rate_c_min)
        return State(
            temp_c=s.temp_c + rate * dt,
            vib_rms=s.vib_rms * (1.0 + 0.02 * dt),
            rate_c_min=rate,
            load_pct=s.load_pct,
        )
    if action == "THROTTLE_20":
        load = max(0.0, s.load_pct - 20.0)
        target_rate = -0.35
        rate = s.rate_c_min + 0.6 * (target_rate - s.rate_c_min)
        return State(
            temp_c=s.temp_c + rate * dt,
            vib_rms=max(0.2, s.vib_rms - 0.18 * dt),
            rate_c_min=rate,
            load_pct=load,
        )
    if action == "COOLING_BOOST":
        load = max(0.0, s.load_pct - 10.0)
        target_rate = -0.85
        rate = s.rate_c_min + 0.6 * (target_rate - s.rate_c_min)
        return State(
            temp_c=s.temp_c + rate * dt,
            vib_rms=max(0.2, s.vib_rms - 0.10 * dt),
            rate_c_min=rate,
            load_pct=load,
        )
    if action == "SHUTDOWN":
        rate = -2.50
        return State(
            temp_c=s.temp_c + rate * dt,
            vib_rms=max(0.0, s.vib_rms - 0.65 * dt),
            rate_c_min=rate,
            load_pct=0.0,
        )
    raise ValueError(f"Unsupported action: {action}")


def rollout_neural_hybrid(
    s0: State,
    action: str,
    predictor: NeuralTransitionEngine,
    steps: int = 5,
    dt: float = 1.0,
    temp_limit_c: float = 95.0,
    vib_limit_rms: float = 3.5,
) -> Dict[str, Any]:
    cur = s0
    phys_cost = ACTION_COSTS[action]
    pred_err_sum = 0.0
    max_t = s0.temp_c
    max_v = s0.vib_rms
    violation = False

    for _ in range(steps):
        nxt = step_fn(cur, action, dt=dt)
        pred_err_sum += predictor.transition_error(cur, action, nxt)

        max_t = max(max_t, nxt.temp_c)
        max_v = max(max_v, nxt.vib_rms)

        if nxt.temp_c > temp_limit_c or nxt.vib_rms > vib_limit_rms:
            violation = True
            phys_cost += 25.0
        phys_cost += max(0.0, nxt.temp_c - 75.0) * 0.4 + max(0.0, nxt.vib_rms - 2.0) * 8.0

        cur = nxt

    avg_pred_err = round(pred_err_sum / max(1, steps), 6)
    neural_penalty = round(predictor.neural_weight * pred_err_sum, 4)
    total_cost = round(phys_cost + neural_penalty, 4)

    return {
        "action": action,
        "cost": total_cost,
        "phys_cost": round(phys_cost, 4),
        "neural_err": avg_pred_err,
        "neural_penalty": neural_penalty,
        "jepa_energy": avg_pred_err,
        "jepa_penalty": neural_penalty,
        "violation": violation,
        "max_t": round(max_t, 4),
        "max_v": round(max_v, 4),
        "end": {
            "temp_c": round(cur.temp_c, 6),
            "vib_rms": round(cur.vib_rms, 6),
            "rate_c_min": round(cur.rate_c_min, 6),
            "load_pct": round(cur.load_pct, 4),
        },
    }


def extract_json(text: str) -> Dict[str, Any]:
    cleaned = re.sub(r"<\|.*?\|>", " ", text)
    cleaned = re.sub(r"^to=\S+\s*", "", cleaned.strip())
    for match in re.finditer(r"\{.*?\}", cleaned, flags=re.DOTALL):
        candidate = match.group(0)
        try:
            parsed = json.loads(candidate)
            if isinstance(parsed, dict) and parsed:
                return parsed
        except json.JSONDecodeError:
            continue
    return {"raw": cleaned.strip()}


def normalize_sql(sql_text: str, best_rollout: Dict[str, Any]) -> str:
    cleaned = re.sub(r"```sql|```", "", sql_text, flags=re.IGNORECASE).strip()
    cleaned = re.sub(
        r"\bFROM\s+table\s+control_log\b",
        "FROM control_log",
        cleaned,
        flags=re.IGNORECASE,
    )
    cleaned = re.sub(
        r"\bFROM\s+control_log\s*\([^)]*\)",
        "FROM control_log",
        cleaned,
        flags=re.IGNORECASE,
    )
    if not re.search(r"\bSELECT\b", cleaned, flags=re.IGNORECASE):
        return (
            f"SELECT action, cost, max_t, max_v FROM control_log "
            f"WHERE action = '{best_rollout['action']}' AND cost = {best_rollout['cost']}"
        )
    return cleaned


def query_gpu_status() -> Tuple[str, Optional[int]]:
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,memory.used", "--format=csv,noheader,nounits"],
            text=True,
            stderr=subprocess.DEVNULL,
        ).splitlines()
        if not out:
            return "RTX", None
        gpu_name, mem_str = [p.strip() for p in out[0].split(",", 1)]
        return gpu_name, int(float(mem_str))
    except Exception:
        return "RTX", None


def ask_llm(
    path: Path,
    system: str,
    user: str,
    n_gpu_layers: int = -1,
    n_ctx: int = 2048,
    n_batch: int = 512,
    max_tokens: int = 400,
    keep_resident: bool = True,
) -> Tuple[str, Optional[int]]:
    key = str(path.resolve())
    if keep_resident and key in _RESIDENT_MODELS:
        llm = _RESIDENT_MODELS[key]
    else:
        llm = Llama(
            model_path=str(path),
            n_gpu_layers=n_gpu_layers,
            n_ctx=n_ctx,
            n_batch=n_batch,
            flash_attn=True,
            verbose=False,
        )
        if keep_resident:
            _RESIDENT_MODELS[key] = llm

    resp = llm.create_chat_completion(
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        temperature=0.0,
        max_tokens=max_tokens,
    )
    out = (resp["choices"][0]["message"]["content"] or "").strip()
    _, vram_used = query_gpu_status()

    if not keep_resident:
        del llm
        gc.collect()

    return out, vram_used


def format_cycle_report(record: Dict[str, Any]) -> str:
    lines: List[str] = []
    cycle_num = int(record.get("cycle", 0))
    best = record.get("best_rollout", {})
    net_meta = record.get("neural_transition", {})

    lines.append("=" * 84)
    lines.append(f"SOMALA • CLOSED-LOOP CYCLE {cycle_num} TELEMETRY REPORT (NEURAL + 5-MODEL GGUF)")
    lines.append("=" * 84)
    lines.append(f"Timestamp (UTC) : {record.get('ts_utc', 'N/A')}")
    lines.append(f"GPU             : {record.get('gpu', 'N/A')}")
    lines.append(f"Cycle Latency   : {record.get('latency_s', 'N/A')}s")
    lines.append(
        f"VRAM (GB)       : peak={format_vram_gb(record.get('vram_used_mib'))} | "
        f"post_cycle={format_vram_gb(record.get('vram_post_cycle_mib'))}"
    )
    lines.append(f"Max Drift       : {record.get('max_drift', 'N/A')}")

    lines.append("")
    lines.append("[Neural Transition Engine]")
    lines.append(
        f"  active=True | device={net_meta.get('device', 'cpu')} | hidden={net_meta.get('hidden_dim', 64)} | "
        f"weight={net_meta.get('neural_weight', 0.25)} | train_mse={net_meta.get('final_loss', 'N/A')} | "
        f"s_norm={net_meta.get('state_signature', 'N/A')}"
    )
    if best:
        lines.append(
            f"  selected_cost   : phys_cost={best.get('phys_cost')} + "
            f"neural_penalty={best.get('neural_penalty')} (avg_err={best.get('neural_err')}) "
            f"-> cost={best.get('cost')}"
        )

    s_in = record.get("state_in", {})
    s_next = record.get("state_next", {})
    lines.append("")
    lines.append("[State Transition]")
    lines.append(
        f"  s_{cycle_num} (in)  : temp_c={s_in.get('temp_c')} C | "
        f"vib_rms={s_in.get('vib_rms')} | rate={s_in.get('rate_c_min')} C/min | load={s_in.get('load_pct')}%"
    )
    lines.append(
        f"  s_{cycle_num + 1} (next): temp_c={s_next.get('temp_c')} C | "
        f"vib_rms={s_next.get('vib_rms')} | rate={s_next.get('rate_c_min')} C/min | load={s_next.get('load_pct')}%"
    )

    models = record.get("models", {})
    hashes = record.get("sha256_16", {})
    drifts = record.get("drifts", {})
    vrams = record.get("vram_by_stage_mib", {})

    lines.append("")
    if cycle_num == 0:
        lines.append("[Stage Models, SHA-256 & VRAM Load]")
        prev_mib: Optional[float] = None
        for stage in ["physical", "perceive", "simulate", "policy", "act"]:
            cur_mib = vrams.get(stage)
            delta_str = ""
            if cur_mib is not None:
                if prev_mib is None:
                    delta_str = f" (+{float(cur_mib) / 1024.0:.2f} GB)"
                else:
                    delta_str = f" ({(float(cur_mib) - prev_mib) / 1024.0:+.2f} GB)"
                prev_mib = float(cur_mib)
            lines.append(
                f"  {stage:<9} | {str(models.get(stage, 'N/A')):<46} | "
                f"sha={str(hashes.get(stage, 'N/A'))} | vram={format_vram_gb(cur_mib):>8}{delta_str:<11} | "
                f"drift={drifts.get(stage, 'N/A')}"
            )
    else:
        lines.append(
            f"[Stage Models & SHA-256 — Resident Pool: {format_vram_gb(record.get('vram_used_mib'))}]"
        )
        for stage in ["physical", "perceive", "simulate", "policy", "act"]:
            lines.append(
                f"  {stage:<9} | {str(models.get(stage, 'N/A')):<46} | "
                f"sha={str(hashes.get(stage, 'N/A'))} | status=warm/resident | "
                f"drift={drifts.get(stage, 'N/A')}"
            )

    lines.append("")
    lines.append("[Neural Hybrid Rollout Evaluation]")
    for r in record.get("rollouts", []):
        act = r.get("action", "N/A")
        tag = " <-- SELECTED" if act == best.get("action") else ""
        end = r.get("end", {})
        lines.append(
            f"  {act:<14} | cost={str(r.get('cost')):<8} | err={str(r.get('neural_err')):<8} | "
            f"viol={str(r.get('violation')):<5} | max_t={str(r.get('max_t')):<7} | max_v={str(r.get('max_v')):<6} | "
            f"end=(T={end.get('temp_c')}, V={end.get('vib_rms')}, dT={end.get('rate_c_min')}, L={end.get('load_pct')}%){tag}"
        )

    pol = record.get("policy", {})
    lines.append("")
    lines.append("[Decision & SQL]")
    lines.append(
        f"  Executed Action : {record.get('executed_action')} "
        f"(cost={best.get('cost')}, neural_err={best.get('neural_err')})"
    )
    lines.append(
        f"  Policy Rationale: {pol.get('rationale', '')} (fallback={pol.get('fallback', False)})"
    )
    lines.append(f"  SQL             : {record.get('sql', 'N/A')}")
    lines.append("=" * 84)
    return "\n".join(lines)


def run_closed_loop(
    initial_state: State = State(temp_c=78.5, vib_rms=2.45, rate_c_min=1.5, load_pct=88.0),
    cycles: int = 3,
    rollout_steps: int = 5,
    dt: float = 1.0,
    hidden_dim: int = 64,
    neural_weight: float = 0.25,
    n_gpu_layers: int = -1,
    keep_resident: bool = True,
    cache_dir: Optional[Path] = None,
    ledger: Path = Path("world_model_neural_closed_loop_telemetry.jsonl"),
) -> List[Dict[str, Any]]:
    paths: Dict[str, Path] = {
        role: get_gguf(repo, cache_dir=cache_dir) for role, repo in SELECTIVE_MODELS.items()
    }
    hashes: Dict[str, str] = {}
    drifts: Dict[str, float] = {}
    variances: Dict[str, float] = {}

    for role, path in paths.items():
        h, d, v = check_drift(path)
        hashes[role] = h
        drifts[role] = d
        variances[role] = v

    predictor = NeuralTransitionEngine(
        hidden_dim=hidden_dim,
        neural_weight=neural_weight,
        seed=42,
    )
    current_state = initial_state
    trajectory: List[Dict[str, Any]] = []
    ledger.parent.mkdir(parents=True, exist_ok=True)

    for cycle_idx in range(cycles):
        t0 = time.perf_counter()
        obs = format_observation(current_state)
        vram_by_stage: Dict[str, Optional[int]] = {}

        # 1. Physical / sensor grounding (Nemotron)
        phys_raw, vram_by_stage["physical"] = ask_llm(
            paths["physical"],
            'Return only one JSON object: {"physical_regime": "...", "thermal_margin_c": 0.0}. Do not repeat input.',
            obs,
            n_gpu_layers=n_gpu_layers,
            max_tokens=160,
            keep_resident=keep_resident,
        )
        phys_json = extract_json(phys_raw)

        # 2. Perception / numeric state encoding (Gemma 4)
        perc_raw, vram_by_stage["perceive"] = ask_llm(
            paths["perceive"],
            'Return only one JSON object with numeric keys: {"temp_c": 0.0, "vib_rms": 0.0, "rate_c_min": 0.0, "load_pct": 0.0}.',
            f"Observation: {obs}\nPhysical: {json.dumps(phys_json)}",
            n_gpu_layers=n_gpu_layers,
            max_tokens=160,
            keep_resident=keep_resident,
        )
        perc_json = extract_json(perc_raw)
        try:
            s_in = State(
                temp_c=float(perc_json["temp_c"]),
                vib_rms=float(perc_json["vib_rms"]),
                rate_c_min=float(perc_json["rate_c_min"]),
                load_pct=float(perc_json["load_pct"]),
            )
        except Exception:
            s_in = current_state

        s_sig = predictor.state_signature(s_in)

        # 3. World dynamics & Neural hybrid rollout evaluation (Muse Glimmer 30B + step_fn + NeuralTransitionEngine)
        rollouts = [
            rollout_neural_hybrid(s_in, act, predictor, steps=rollout_steps, dt=dt)
            for act in VALID_ACTIONS
        ]
        best_rollout = min(rollouts, key=lambda r: r["cost"])
        compact_summary = [
            {
                "action": r["action"],
                "cost": r["cost"],
                "neural_err": r["neural_err"],
                "violation": r["violation"],
                "max_t": r["max_t"],
                "max_v": r["max_v"],
            }
            for r in rollouts
        ]
        sim_raw, vram_by_stage["simulate"] = ask_llm(
            paths["simulate"],
            'Return only one concise JSON object: {"sim_summary": "short summary"}. Do not echo input.',
            json.dumps(compact_summary),
            n_gpu_layers=n_gpu_layers,
            max_tokens=300,
            keep_resident=keep_resident,
        )
        sim_json = extract_json(sim_raw)
        if "raw" in sim_json or "sim_summary" not in sim_json:
            sim_json = {
                "sim_summary": (
                    f"Lowest-cost safe option is {best_rollout['action']} "
                    f"(cost={best_rollout['cost']}, neural_err={best_rollout['neural_err']}, "
                    f"max_t={best_rollout['max_t']}C, max_v={best_rollout['max_v']} rms)."
                ),
                "fallback": True,
            }

        # 4. Policy selection (DeepSeek-MoE 16B grounded by lowest hybrid rollout cost)
        policy_raw, vram_by_stage["policy"] = ask_llm(
            paths["policy"],
            'Return only one JSON object: {"selected_action": "...", "rationale": "..."}.',
            f"Candidate rollouts: {json.dumps(compact_summary)}\nLowest-cost rollout: {best_rollout['action']}",
            n_gpu_layers=n_gpu_layers,
            max_tokens=300,
            keep_resident=keep_resident,
        )
        policy_json = extract_json(policy_raw)
        selected_action = str(policy_json.get("selected_action", "")).strip().upper()
        if selected_action not in VALID_ACTIONS:
            selected_action = best_rollout["action"]
            policy_json = {
                "selected_action": selected_action,
                "rationale": f"Deterministic fallback to lowest-cost rollout (cost={best_rollout['cost']}).",
                "fallback": True,
            }

        # Advance state from s_k to s_{k+1} using the selected action
        next_state = step_fn(s_in, selected_action, dt=dt)

        # 5. Action / SQL emission (Mistral 7B Text-to-SQL)
        sql_raw, vram_by_stage["act"] = ask_llm(
            paths["act"],
            "Output ONLY a single valid SQL SELECT query from table control_log (columns: action, cost, max_t, max_v).",
            json.dumps({"cycle": cycle_idx, "action": selected_action, "rollout": best_rollout}),
            n_gpu_layers=n_gpu_layers,
            max_tokens=200,
            keep_resident=keep_resident,
        )
        sql_query = normalize_sql(sql_raw, best_rollout)

        gpu_name, post_cycle_vram_mib = query_gpu_status()
        active_vrams = [v for v in vram_by_stage.values() if v is not None]
        peak_stage_vram_mib = max(active_vrams) if active_vrams else post_cycle_vram_mib

        s_in_clean = {
            "temp_c": round(s_in.temp_c, 6),
            "vib_rms": round(s_in.vib_rms, 6),
            "rate_c_min": round(s_in.rate_c_min, 6),
            "load_pct": round(s_in.load_pct, 4),
        }
        s_next_clean = {
            "temp_c": round(next_state.temp_c, 6),
            "vib_rms": round(next_state.vib_rms, 6),
            "rate_c_min": round(next_state.rate_c_min, 6),
            "load_pct": round(next_state.load_pct, 4),
        }

        record: Dict[str, Any] = {
            "ts_utc": datetime.now(timezone.utc).isoformat(),
            "cycle": cycle_idx,
            "gpu": gpu_name,
            "vram_used_mib": peak_stage_vram_mib,
            "vram_post_cycle_mib": post_cycle_vram_mib,
            "vram_by_stage_mib": vram_by_stage,
            "models": {role: p.name for role, p in paths.items()},
            "sha256_16": hashes,
            "drifts": drifts,
            "variances": variances,
            "max_drift": max(drifts.values()),
            "neural_transition": {
                "enabled": True,
                "device": str(predictor.device),
                "hidden_dim": predictor.hidden_dim,
                "neural_weight": predictor.neural_weight,
                "final_loss": predictor.final_loss,
                "state_signature": s_sig,
            },
            "observation_in": obs,
            "state_in": s_in_clean,
            "state_next": s_next_clean,
            "executed_action": selected_action,
            "physical": phys_json,
            "perception": perc_json,
            "simulation": sim_json,
            "policy": policy_json,
            "best_rollout": best_rollout,
            "rollouts": rollouts,
            "sql": sql_query,
            "latency_s": round(time.perf_counter() - t0, 3),
        }

        with ledger.open("a", encoding="utf-8") as f:
            f.write(json.dumps(record) + "\n")

        print(format_cycle_report(record))
        trajectory.append(record)
        current_state = next_state

    return trajectory


def main() -> None:
    parser = argparse.ArgumentParser(description="SOMALA 5-Model + Neural Transition Closed-Loop World Model")
    parser.add_argument("--cycles", type=int, default=3)
    parser.add_argument("--steps", type=int, default=5)
    parser.add_argument("--dt", type=float, default=1.0)
    parser.add_argument("--hidden-dim", type=int, default=64)
    parser.add_argument("--neural-weight", type=float, default=0.25)
    parser.add_argument("--temp-c", type=float, default=78.5)
    parser.add_argument("--vib-rms", type=float, default=2.45)
    parser.add_argument("--rate-c-min", type=float, default=1.5)
    parser.add_argument("--load-pct", type=float, default=88.0)
    parser.add_argument("--n-gpu-layers", type=int, default=-1)
    parser.add_argument("--no-resident", action="store_true")
    parser.add_argument("--cache-dir", type=Path, default=None)
    parser.add_argument("--ledger", type=Path, default=Path("world_model_neural_closed_loop_telemetry.jsonl"))
    args, _ = parser.parse_known_args()

    init_s = State(
        temp_c=args.temp_c,
        vib_rms=args.vib_rms,
        rate_c_min=args.rate_c_min,
        load_pct=args.load_pct,
    )
    run_closed_loop(
        initial_state=init_s,
        cycles=args.cycles,
        rollout_steps=args.steps,
        dt=args.dt,
        hidden_dim=args.hidden_dim,
        neural_weight=args.neural_weight,
        n_gpu_layers=args.n_gpu_layers,
        keep_resident=not args.no_resident,
        cache_dir=args.cache_dir,
        ledger=args.ledger,
    )


if __name__ == "__main__":
    run_closed_loop()

llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)
llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)
llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)


SOMALA • CLOSED-LOOP CYCLE 0 TELEMETRY REPORT (NEURAL + 5-MODEL GGUF)
Timestamp (UTC) : 2026-11-21T16:22:34.996447+00:00
GPU             : NVIDIA RTX PRO 6000 Blackwell Server Edition
Cycle Latency   : 11.507s
VRAM (GB)       : peak=64.01 GB | post_cycle=64.01 GB
Max Drift       : 0.0

[Neural Transition Engine]
  active=True | device=cuda | hidden=64 | weight=0.25 | train_mse=0.008031 | s_norm=[0.175, 0.3, 0.6, 0.88]
  selected_cost   : phys_cost=11.9211 + neural_penalty=0.0006 (avg_err=0.000491) -> cost=11.9217

[State Transition]
  s_0 (in)  : temp_c=78.5 C | vib_rms=2.45 | rate=1.5 C/min | load=88.0%
  s_1 (next): temp_c=78.89 C | vib_rms=2.27 | rate=0.39 C/min | load=68.0%

[Stage Models, SHA-256 & VRAM Load]
  physical  | nemotron-3.5-bolt-topo-governed-Q4_K_M.gguf    | sha=7e59d0116378b6d8 | vram= 4.25 GB (+4.25 GB) | drift=0.0
  perceive  | Gemma-4-26B-A4B-alpaca-topo-epoch-1-Q8_0.gguf  | sha=bfe4b3f85a110f1b | vram=30.28 GB (+26.03 GB) | drift=0.0
  simulate  | muse-glimmer-30

## CONSOLIDATION

In [8]:
#!/usr/bin/env python3
"""
SOMALA • Closed-Loop 5-Model Selective GGUF World Model + Trained PyTorch JEPA Layer
Trajectory: s_0 -> PyTorch JEPA hybrid rollouts/policy/SQL -> s_1 = step_fn(s_0, a_0) -> s_2 -> ...
"""

import argparse
import gc
import hashlib
import json
import math
import re
import subprocess
import time
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from huggingface_hub import HfApi, hf_hub_download
from llama_cpp import Llama

ANCHORS: List[int] = [2, 3, 5, 7, 11, 13]

SELECTIVE_MODELS: Dict[str, str] = {
    "physical": "frankmorales2020/nemotron-3.5-bolt-e4b-topo-governed-fineweb-gguf",
    "perceive": "frankmorales2020/Gemma-4-26B-A4B-alpaca-topo-epoch-1-Q8_0-GGUF",
    "simulate": "frankmorales2020/muse-glimmer-30b-topo-governed-cbp-fineweb-gguf",
    "policy": "frankmorales2020/deepseek-moe-16b-topo-cbp-merged-gguf",
    "act": "frankmorales2020/mistral-7b-text-to-sql-topo-Q8_0-GGUF",
}

VALID_ACTIONS: List[str] = ["NO_OP", "THROTTLE_20", "COOLING_BOOST", "SHUTDOWN"]

ACTION_COSTS: Dict[str, float] = {
    "NO_OP": 0.0,
    "THROTTLE_20": 2.0,
    "COOLING_BOOST": 3.0,
    "SHUTDOWN": 18.0,
}

_RESIDENT_MODELS: Dict[str, Llama] = {}


@dataclass
class State:
    temp_c: float
    vib_rms: float
    rate_c_min: float
    load_pct: float


def _action_one_hot_np(action: str) -> np.ndarray:
    vec = np.zeros(len(VALID_ACTIONS), dtype=np.float32)
    if action in VALID_ACTIONS:
        vec[VALID_ACTIONS.index(action)] = 1.0
    return vec


class LatentEncoder(nn.Module):
    """Encodes normalized physical state (4) -> unit-norm latent vector z (latent_dim)."""

    def __init__(self, in_dim: int = 4, hidden_dim: int = 64, latent_dim: int = 16) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, latent_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        z = self.net(x)
        return F.normalize(z, p=2, dim=-1, eps=1e-8)


class LatentPredictor(nn.Module):
    """Predicts next-step unit-norm latent z_hat_{t+1} from [z_t, action_oh]."""

    def __init__(self, latent_dim: int = 16, n_actions: int = 4, hidden_dim: int = 64) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim + n_actions, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, latent_dim),
        )

    def forward(self, z_t: torch.Tensor, action_oh: torch.Tensor) -> torch.Tensor:
        h = torch.cat([z_t, action_oh], dim=-1)
        z_hat = z_t + self.net(h)
        return F.normalize(z_hat, p=2, dim=-1, eps=1e-8)


class PyTorchJEPAEngine:
    """
    Trained PyTorch-native JEPA encoder + predictor fitted on rollout transitions.
    Uses cosine prediction energy (1 - cos_sim) + variance regularization to prevent collapse.
    """

    def __init__(
        self,
        latent_dim: int = 16,
        hidden_dim: int = 64,
        train_samples: int = 1024,
        epochs: int = 80,
        lr: float = 2e-3,
        jepa_weight: float = 0.25,
        var_reg_weight: float = 0.05,
        seed: int = 42,
        device: Optional[str] = None,
    ) -> None:
        self.latent_dim = latent_dim
        self.hidden_dim = hidden_dim
        self.jepa_weight = jepa_weight
        self.var_reg_weight = var_reg_weight
        self.device = (
            torch.device("cuda" if torch.cuda.is_available() else "cpu")
            if device is None
            else torch.device(device)
        )

        torch.manual_seed(seed)
        rng = np.random.default_rng(seed)

        self.encoder = LatentEncoder(4, hidden_dim, latent_dim).to(self.device)
        self.predictor = LatentPredictor(latent_dim, len(VALID_ACTIONS), hidden_dim).to(self.device)

        self.final_loss: float = 0.0
        self._fit_on_transitions(rng=rng, samples=train_samples, epochs=epochs, lr=lr)

    @staticmethod
    def _norm_vec(s: State) -> np.ndarray:
        return np.array(
            [
                (s.temp_c - 75.0) / 20.0,
                (s.vib_rms - 2.0) / 1.5,
                s.rate_c_min / 2.5,
                s.load_pct / 100.0,
            ],
            dtype=np.float32,
        )

    @staticmethod
    def _variance_hinge_loss(z: torch.Tensor, target_std: float = 0.20) -> torch.Tensor:
        std = torch.sqrt(z.var(dim=0, unbiased=False) + 1e-6)
        return F.relu(target_std - std).mean()

    def _fit_on_transitions(
        self,
        rng: np.random.Generator,
        samples: int,
        epochs: int,
        lr: float,
    ) -> None:
        s_cur_list, act_list, s_nxt_list = [], [], []
        for _ in range(samples):
            s0 = State(
                temp_c=float(rng.uniform(65.0, 98.0)),
                vib_rms=float(rng.uniform(0.5, 4.0)),
                rate_c_min=float(rng.uniform(-2.5, 2.5)),
                load_pct=float(rng.uniform(0.0, 100.0)),
            )
            act = str(rng.choice(VALID_ACTIONS))
            s1 = step_fn(s0, act, dt=1.0)
            s_cur_list.append(self._norm_vec(s0))
            act_list.append(_action_one_hot_np(act))
            s_nxt_list.append(self._norm_vec(s1))

        x_cur = torch.tensor(np.stack(s_cur_list), dtype=torch.float32, device=self.device)
        a_oh = torch.tensor(np.stack(act_list), dtype=torch.float32, device=self.device)
        x_nxt = torch.tensor(np.stack(s_nxt_list), dtype=torch.float32, device=self.device)

        params = list(self.encoder.parameters()) + list(self.predictor.parameters())
        opt = torch.optim.AdamW(params, lr=lr)

        self.encoder.train()
        self.predictor.train()
        for _ in range(epochs):
            opt.zero_grad()
            z_t = self.encoder(x_cur)
            with torch.no_grad():
                z_target = self.encoder(x_nxt)

            z_hat = self.predictor(z_t, a_oh)
            cos_energy = (1.0 - F.cosine_similarity(z_hat, z_target, dim=-1)).mean()
            var_loss = self._variance_hinge_loss(z_t)
            loss = cos_energy + self.var_reg_weight * var_loss
            loss.backward()
            opt.step()

        self.encoder.eval()
        self.predictor.eval()
        with torch.no_grad():
            z_t = self.encoder(x_cur)
            z_target = self.encoder(x_nxt)
            z_hat = self.predictor(z_t, a_oh)
            self.final_loss = round(
                float((1.0 - F.cosine_similarity(z_hat, z_target, dim=-1)).mean().item()),
                6,
            )

    def encode(self, s: State) -> np.ndarray:
        x = torch.tensor(self._norm_vec(s), dtype=torch.float32, device=self.device).unsqueeze(0)
        self.encoder.eval()
        with torch.no_grad():
            return self.encoder(x).squeeze(0).cpu().numpy().astype(np.float32)

    def predict_next_latent(self, z_t: np.ndarray, action: str) -> np.ndarray:
        zt = torch.tensor(z_t, dtype=torch.float32, device=self.device).unsqueeze(0)
        ah = torch.tensor(_action_one_hot_np(action), dtype=torch.float32, device=self.device).unsqueeze(0)
        self.predictor.eval()
        with torch.no_grad():
            return self.predictor(zt, ah).squeeze(0).cpu().numpy().astype(np.float32)

    @staticmethod
    def latent_energy(z_hat: np.ndarray, z_target: np.ndarray) -> float:
        cos_sim = float(np.clip(np.dot(z_hat, z_target), -1.0, 1.0))
        return round(1.0 - cos_sim, 6)


def format_observation(s: State, asset: str = "Bearing B4") -> str:
    return (
        f"{asset}: temp={s.temp_c:.2f}C, vib={s.vib_rms:.3f} rms, "
        f"dT/dt={s.rate_c_min:+.3f}C/min, load={s.load_pct:.1f}%."
    )


def format_vram_gb(mib: Optional[float | int]) -> str:
    if mib is None:
        return "N/A"
    return f"{float(mib) / 1024.0:.2f} GB"


def get_gguf(repo_id: str, cache_dir: Optional[Path] = None) -> Path:
    api = HfApi()
    files = [f for f in api.list_repo_files(repo_id) if f.lower().endswith(".gguf")]
    if not files:
        raise FileNotFoundError(f"No .gguf file found in {repo_id}")
    return Path(
        hf_hub_download(
            repo_id=repo_id,
            filename=files[0],
            cache_dir=str(cache_dir) if cache_dir else None,
        )
    )


def check_drift(path: Path) -> Tuple[str, float, float]:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(chunk)
        rows = []
        for p in ANCHORS:
            f.seek(p * 4096)
            buf = f.read(64)
            if not buf:
                continue
            rows.append(np.frombuffer(buf, dtype=np.uint8).astype(np.float64))
    if not rows:
        raise ValueError(f"Unable to read anchor slices from {path}")
    mat = np.vstack(rows)
    ref = mat.copy()
    drift = float(np.max(np.abs(mat - ref)))
    var = float(np.var(mat - ref))
    if not (math.isfinite(drift) and math.isfinite(var)):
        raise ValueError("Non-finite drift/variance metric computed.")
    return h.hexdigest()[:16], drift, var


def step_fn(s: State, action: str, dt: float = 1.0) -> State:
    if action == "NO_OP":
        target_rate = s.rate_c_min + 0.15 * max(0.0, (s.load_pct - 70.0) / 30.0)
        rate = s.rate_c_min + 0.5 * (target_rate - s.rate_c_min)
        return State(
            temp_c=s.temp_c + rate * dt,
            vib_rms=s.vib_rms * (1.0 + 0.02 * dt),
            rate_c_min=rate,
            load_pct=s.load_pct,
        )
    if action == "THROTTLE_20":
        load = max(0.0, s.load_pct - 20.0)
        target_rate = -0.35
        rate = s.rate_c_min + 0.6 * (target_rate - s.rate_c_min)
        return State(
            temp_c=s.temp_c + rate * dt,
            vib_rms=max(0.2, s.vib_rms - 0.18 * dt),
            rate_c_min=rate,
            load_pct=load,
        )
    if action == "COOLING_BOOST":
        load = max(0.0, s.load_pct - 10.0)
        target_rate = -0.85
        rate = s.rate_c_min + 0.6 * (target_rate - s.rate_c_min)
        return State(
            temp_c=s.temp_c + rate * dt,
            vib_rms=max(0.2, s.vib_rms - 0.10 * dt),
            rate_c_min=rate,
            load_pct=load,
        )
    if action == "SHUTDOWN":
        rate = -2.50
        return State(
            temp_c=s.temp_c + rate * dt,
            vib_rms=max(0.0, s.vib_rms - 0.65 * dt),
            rate_c_min=rate,
            load_pct=0.0,
        )
    raise ValueError(f"Unsupported action: {action}")


def rollout_jepa_hybrid(
    s0: State,
    action: str,
    jepa: PyTorchJEPAEngine,
    steps: int = 5,
    dt: float = 1.0,
    temp_limit_c: float = 95.0,
    vib_limit_rms: float = 3.5,
) -> Dict[str, Any]:
    cur = s0
    z_cur = jepa.encode(cur)
    phys_cost = ACTION_COSTS[action]
    latent_energy_sum = 0.0
    max_t = s0.temp_c
    max_v = s0.vib_rms
    violation = False

    for _ in range(steps):
        z_pred = jepa.predict_next_latent(z_cur, action)
        nxt = step_fn(cur, action, dt=dt)
        z_actual = jepa.encode(nxt)

        latent_energy_sum += jepa.latent_energy(z_pred, z_actual)
        max_t = max(max_t, nxt.temp_c)
        max_v = max(max_v, nxt.vib_rms)

        if nxt.temp_c > temp_limit_c or nxt.vib_rms > vib_limit_rms:
            violation = True
            phys_cost += 25.0
        phys_cost += max(0.0, nxt.temp_c - 75.0) * 0.4 + max(0.0, nxt.vib_rms - 2.0) * 8.0

        cur = nxt
        z_cur = z_actual

    avg_latent_energy = round(latent_energy_sum / max(1, steps), 6)
    jepa_penalty = round(jepa.jepa_weight * latent_energy_sum, 4)
    total_cost = round(phys_cost + jepa_penalty, 4)

    return {
        "action": action,
        "cost": total_cost,
        "phys_cost": round(phys_cost, 4),
        "jepa_energy": avg_latent_energy,
        "jepa_penalty": jepa_penalty,
        "violation": violation,
        "max_t": round(max_t, 4),
        "max_v": round(max_v, 4),
        "end": {
            "temp_c": round(cur.temp_c, 6),
            "vib_rms": round(cur.vib_rms, 6),
            "rate_c_min": round(cur.rate_c_min, 6),
            "load_pct": round(cur.load_pct, 4),
        },
    }


def extract_json(text: str) -> Dict[str, Any]:
    cleaned = re.sub(r"<\|.*?\|>", " ", text)
    cleaned = re.sub(r"^to=\S+\s*", "", cleaned.strip())
    for match in re.finditer(r"\{.*?\}", cleaned, flags=re.DOTALL):
        candidate = match.group(0)
        try:
            parsed = json.loads(candidate)
            if isinstance(parsed, dict) and parsed:
                return parsed
        except json.JSONDecodeError:
            continue
    return {"raw": cleaned.strip()}


def normalize_sql(sql_text: str, best_rollout: Dict[str, Any]) -> str:
    cleaned = re.sub(r"```sql|```", "", sql_text, flags=re.IGNORECASE).strip()
    cleaned = re.sub(
        r"\bFROM\s+table\s+control_log\b",
        "FROM control_log",
        cleaned,
        flags=re.IGNORECASE,
    )
    cleaned = re.sub(
        r"\bFROM\s+control_log\s*\([^)]*\)",
        "FROM control_log",
        cleaned,
        flags=re.IGNORECASE,
    )
    if not re.search(r"\bSELECT\b", cleaned, flags=re.IGNORECASE):
        return (
            f"SELECT action, cost, max_t, max_v FROM control_log "
            f"WHERE action = '{best_rollout['action']}' AND cost = {best_rollout['cost']}"
        )
    return cleaned


def query_gpu_status() -> Tuple[str, Optional[int]]:
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,memory.used", "--format=csv,noheader,nounits"],
            text=True,
            stderr=subprocess.DEVNULL,
        ).splitlines()
        if not out:
            return "RTX", None
        gpu_name, mem_str = [p.strip() for p in out[0].split(",", 1)]
        return gpu_name, int(float(mem_str))
    except Exception:
        return "RTX", None


def ask_llm(
    path: Path,
    system: str,
    user: str,
    n_gpu_layers: int = -1,
    n_ctx: int = 2048,
    n_batch: int = 512,
    max_tokens: int = 400,
    keep_resident: bool = True,
) -> Tuple[str, Optional[int]]:
    key = str(path.resolve())
    if keep_resident and key in _RESIDENT_MODELS:
        llm = _RESIDENT_MODELS[key]
    else:
        llm = Llama(
            model_path=str(path),
            n_gpu_layers=n_gpu_layers,
            n_ctx=n_ctx,
            n_batch=n_batch,
            flash_attn=True,
            verbose=False,
        )
        if keep_resident:
            _RESIDENT_MODELS[key] = llm

    resp = llm.create_chat_completion(
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        temperature=0.0,
        max_tokens=max_tokens,
    )
    out = (resp["choices"][0]["message"]["content"] or "").strip()
    _, vram_used = query_gpu_status()

    if not keep_resident:
        del llm
        gc.collect()

    return out, vram_used


def format_cycle_report(record: Dict[str, Any]) -> str:
    lines: List[str] = []
    cycle_num = int(record.get("cycle", 0))
    best = record.get("best_rollout", {})
    jepa_meta = record.get("jepa", {})

    lines.append("=" * 84)
    lines.append(f"SOMALA • CLOSED-LOOP CYCLE {cycle_num} TELEMETRY REPORT (PYTORCH JEPA + 5-MODEL GGUF)")
    lines.append("=" * 84)
    lines.append(f"Timestamp (UTC) : {record.get('ts_utc', 'N/A')}")
    lines.append(f"GPU             : {record.get('gpu', 'N/A')}")
    lines.append(f"Cycle Latency   : {record.get('latency_s', 'N/A')}s")
    lines.append(
        f"VRAM (GB)       : peak={format_vram_gb(record.get('vram_used_mib'))} | "
        f"post_cycle={format_vram_gb(record.get('vram_post_cycle_mib'))}"
    )
    lines.append(f"Max Drift       : {record.get('max_drift', 'N/A')}")

    lines.append("")
    lines.append("[PyTorch JEPA Latent Engine]")
    lines.append(
        f"  active={jepa_meta.get('enabled', True)} | device={jepa_meta.get('device', 'cuda')} | "
        f"dim={jepa_meta.get('latent_dim', 16)} | weight={jepa_meta.get('jepa_weight', 0.25)} | "
        f"train_cos_e={jepa_meta.get('final_loss', 'N/A')} | z_norm={record.get('jepa_state_norm', 'N/A')} | "
        f"z_in={jepa_meta.get('z_in_preview', 'N/A')}"
    )
    if best:
        lines.append(
            f"  selected_cost   : phys_cost={best.get('phys_cost')} + "
            f"jepa_penalty={best.get('jepa_penalty')} (avg_e={best.get('jepa_energy')}) "
            f"-> cost={best.get('cost')}"
        )

    s_in = record.get("state_in", {})
    s_next = record.get("state_next", {})
    lines.append("")
    lines.append("[State Transition]")
    lines.append(
        f"  s_{cycle_num} (in)  : temp_c={s_in.get('temp_c')} C | "
        f"vib_rms={s_in.get('vib_rms')} | rate={s_in.get('rate_c_min')} C/min | load={s_in.get('load_pct')}%"
    )
    lines.append(
        f"  s_{cycle_num + 1} (next): temp_c={s_next.get('temp_c')} C | "
        f"vib_rms={s_next.get('vib_rms')} | rate={s_next.get('rate_c_min')} C/min | load={s_next.get('load_pct')}%"
    )

    models = record.get("models", {})
    hashes = record.get("sha256_16", {})
    drifts = record.get("drifts", {})
    vrams = record.get("vram_by_stage_mib", {})

    lines.append("")
    if cycle_num == 0:
        lines.append("[Stage Models, SHA-256 & VRAM Load]")
        prev_mib: Optional[float] = None
        for stage in ["physical", "perceive", "simulate", "policy", "act"]:
            cur_mib = vrams.get(stage)
            delta_str = ""
            if cur_mib is not None:
                if prev_mib is None:
                    delta_str = f" (+{float(cur_mib) / 1024.0:.2f} GB)"
                else:
                    delta_str = f" ({(float(cur_mib) - prev_mib) / 1024.0:+.2f} GB)"
                prev_mib = float(cur_mib)
            lines.append(
                f"  {stage:<9} | {str(models.get(stage, 'N/A')):<46} | "
                f"sha={str(hashes.get(stage, 'N/A'))} | vram={format_vram_gb(cur_mib):>8}{delta_str:<11} | "
                f"drift={drifts.get(stage, 'N/A')}"
            )
    else:
        lines.append(
            f"[Stage Models & SHA-256 — Resident Pool: {format_vram_gb(record.get('vram_used_mib'))}]"
        )
        for stage in ["physical", "perceive", "simulate", "policy", "act"]:
            lines.append(
                f"  {stage:<9} | {str(models.get(stage, 'N/A')):<46} | "
                f"sha={str(hashes.get(stage, 'N/A'))} | status=warm/resident | "
                f"drift={drifts.get(stage, 'N/A')}"
            )

    lines.append("")
    lines.append("[PyTorch JEPA Hybrid Rollout Evaluation]")
    for r in record.get("rollouts", []):
        act = r.get("action", "N/A")
        tag = " <-- SELECTED" if act == best.get("action") else ""
        end = r.get("end", {})
        lines.append(
            f"  {act:<14} | cost={str(r.get('cost')):<8} | jepa_e={str(r.get('jepa_energy')):<8} | "
            f"viol={str(r.get('violation')):<5} | max_t={str(r.get('max_t')):<7} | max_v={str(r.get('max_v')):<6} | "
            f"end=(T={end.get('temp_c')}, V={end.get('vib_rms')}, dT={end.get('rate_c_min')}, L={end.get('load_pct')}%){tag}"
        )

    pol = record.get("policy", {})
    lines.append("")
    lines.append("[Decision & SQL]")
    lines.append(
        f"  Executed Action : {record.get('executed_action')} "
        f"(cost={best.get('cost')}, jepa_energy={best.get('jepa_energy')})"
    )
    lines.append(
        f"  Policy Rationale: {pol.get('rationale', '')} (fallback={pol.get('fallback', False)})"
    )
    lines.append(f"  SQL             : {record.get('sql', 'N/A')}")
    lines.append("=" * 84)
    return "\n".join(lines)


def run_closed_loop(
    initial_state: State = State(temp_c=78.5, vib_rms=2.45, rate_c_min=1.5, load_pct=88.0),
    cycles: int = 3,
    rollout_steps: int = 5,
    dt: float = 1.0,
    latent_dim: int = 16,
    hidden_dim: int = 64,
    jepa_weight: float = 0.25,
    n_gpu_layers: int = -1,
    keep_resident: bool = True,
    cache_dir: Optional[Path] = None,
    ledger: Path = Path("world_model_pytorch_jepa_closed_loop_telemetry.jsonl"),
) -> List[Dict[str, Any]]:
    paths: Dict[str, Path] = {
        role: get_gguf(repo, cache_dir=cache_dir) for role, repo in SELECTIVE_MODELS.items()
    }
    hashes: Dict[str, str] = {}
    drifts: Dict[str, float] = {}
    variances: Dict[str, float] = {}

    for role, path in paths.items():
        h, d, v = check_drift(path)
        hashes[role] = h
        drifts[role] = d
        variances[role] = v

    jepa = PyTorchJEPAEngine(
        latent_dim=latent_dim,
        hidden_dim=hidden_dim,
        jepa_weight=jepa_weight,
        seed=42,
    )
    current_state = initial_state
    trajectory: List[Dict[str, Any]] = []
    ledger.parent.mkdir(parents=True, exist_ok=True)

    for cycle_idx in range(cycles):
        t0 = time.perf_counter()
        obs = format_observation(current_state)
        vram_by_stage: Dict[str, Optional[int]] = {}

        # 1. Physical / sensor grounding (Nemotron)
        phys_raw, vram_by_stage["physical"] = ask_llm(
            paths["physical"],
            'Return only one JSON object: {"physical_regime": "...", "thermal_margin_c": 0.0}. Do not repeat input.',
            obs,
            n_gpu_layers=n_gpu_layers,
            max_tokens=160,
            keep_resident=keep_resident,
        )
        phys_json = extract_json(phys_raw)

        # 2. Perception / numeric state encoding (Gemma 4)
        perc_raw, vram_by_stage["perceive"] = ask_llm(
            paths["perceive"],
            'Return only one JSON object with numeric keys: {"temp_c": 0.0, "vib_rms": 0.0, "rate_c_min": 0.0, "load_pct": 0.0}.',
            f"Observation: {obs}\nPhysical: {json.dumps(phys_json)}",
            n_gpu_layers=n_gpu_layers,
            max_tokens=160,
            keep_resident=keep_resident,
        )
        perc_json = extract_json(perc_raw)
        try:
            s_in = State(
                temp_c=float(perc_json["temp_c"]),
                vib_rms=float(perc_json["vib_rms"]),
                rate_c_min=float(perc_json["rate_c_min"]),
                load_pct=float(perc_json["load_pct"]),
            )
        except Exception:
            s_in = current_state

        z_in = jepa.encode(s_in)
        z_in_norm = round(float(np.linalg.norm(z_in)), 6)

        # 3. World dynamics & PyTorch JEPA hybrid rollout evaluation
        rollouts = [
            rollout_jepa_hybrid(s_in, act, jepa, steps=rollout_steps, dt=dt)
            for act in VALID_ACTIONS
        ]
        best_rollout = min(rollouts, key=lambda r: r["cost"])
        compact_summary = [
            {
                "action": r["action"],
                "cost": r["cost"],
                "jepa_energy": r["jepa_energy"],
                "violation": r["violation"],
                "max_t": r["max_t"],
                "max_v": r["max_v"],
            }
            for r in rollouts
        ]
        sim_raw, vram_by_stage["simulate"] = ask_llm(
            paths["simulate"],
            'Return only one concise JSON object: {"sim_summary": "short summary"}. Do not echo input.',
            json.dumps(compact_summary),
            n_gpu_layers=n_gpu_layers,
            max_tokens=300,
            keep_resident=keep_resident,
        )
        sim_json = extract_json(sim_raw)
        if "raw" in sim_json or "sim_summary" not in sim_json:
            sim_json = {
                "sim_summary": (
                    f"Lowest-cost safe option is {best_rollout['action']} "
                    f"(cost={best_rollout['cost']}, jepa_energy={best_rollout['jepa_energy']}, "
                    f"max_t={best_rollout['max_t']}C, max_v={best_rollout['max_v']} rms)."
                ),
                "fallback": True,
            }

        # 4. Policy selection (DeepSeek-MoE 16B grounded by lowest hybrid rollout cost)
        policy_raw, vram_by_stage["policy"] = ask_llm(
            paths["policy"],
            'Return only one JSON object: {"selected_action": "...", "rationale": "..."}.',
            f"Candidate rollouts: {json.dumps(compact_summary)}\nLowest-cost rollout: {best_rollout['action']}",
            n_gpu_layers=n_gpu_layers,
            max_tokens=300,
            keep_resident=keep_resident,
        )
        policy_json = extract_json(policy_raw)
        selected_action = str(policy_json.get("selected_action", "")).strip().upper()
        if selected_action not in VALID_ACTIONS:
            selected_action = best_rollout["action"]
            policy_json = {
                "selected_action": selected_action,
                "rationale": f"Deterministic fallback to lowest-cost rollout (cost={best_rollout['cost']}).",
                "fallback": True,
            }

        # Advance state from s_k to s_{k+1} using the selected action
        next_state = step_fn(s_in, selected_action, dt=dt)

        # 5. Action / SQL emission (Mistral 7B Text-to-SQL)
        sql_raw, vram_by_stage["act"] = ask_llm(
            paths["act"],
            "Output ONLY a single valid SQL SELECT query from table control_log (columns: action, cost, max_t, max_v).",
            json.dumps({"cycle": cycle_idx, "action": selected_action, "rollout": best_rollout}),
            n_gpu_layers=n_gpu_layers,
            max_tokens=200,
            keep_resident=keep_resident,
        )
        sql_query = normalize_sql(sql_raw, best_rollout)

        gpu_name, post_cycle_vram_mib = query_gpu_status()
        active_vrams = [v for v in vram_by_stage.values() if v is not None]
        peak_stage_vram_mib = max(active_vrams) if active_vrams else post_cycle_vram_mib

        s_in_clean = {
            "temp_c": round(s_in.temp_c, 6),
            "vib_rms": round(s_in.vib_rms, 6),
            "rate_c_min": round(s_in.rate_c_min, 6),
            "load_pct": round(s_in.load_pct, 4),
        }
        s_next_clean = {
            "temp_c": round(next_state.temp_c, 6),
            "vib_rms": round(next_state.vib_rms, 6),
            "rate_c_min": round(next_state.rate_c_min, 6),
            "load_pct": round(next_state.load_pct, 4),
        }

        record: Dict[str, Any] = {
            "ts_utc": datetime.now(timezone.utc).isoformat(),
            "cycle": cycle_idx,
            "gpu": gpu_name,
            "vram_used_mib": peak_stage_vram_mib,
            "vram_post_cycle_mib": post_cycle_vram_mib,
            "vram_by_stage_mib": vram_by_stage,
            "models": {role: p.name for role, p in paths.items()},
            "sha256_16": hashes,
            "drifts": drifts,
            "variances": variances,
            "max_drift": max(drifts.values()),
            "jepa_state_norm": z_in_norm,
            "jepa": {
                "enabled": True,
                "backend": "pytorch",
                "device": str(jepa.device),
                "latent_dim": jepa.latent_dim,
                "hidden_dim": jepa.hidden_dim,
                "jepa_weight": jepa.jepa_weight,
                "final_loss": jepa.final_loss,
                "z_in_preview": [round(float(x), 3) for x in z_in[:4]],
            },
            "observation_in": obs,
            "state_in": s_in_clean,
            "state_next": s_next_clean,
            "executed_action": selected_action,
            "physical": phys_json,
            "perception": perc_json,
            "simulation": sim_json,
            "policy": policy_json,
            "best_rollout": best_rollout,
            "rollouts": rollouts,
            "sql": sql_query,
            "latency_s": round(time.perf_counter() - t0, 3),
        }

        with ledger.open("a", encoding="utf-8") as f:
            f.write(json.dumps(record) + "\n")

        print(format_cycle_report(record))
        trajectory.append(record)
        current_state = next_state

    return trajectory


def main() -> None:
    parser = argparse.ArgumentParser(description="SOMALA 5-Model + Trained PyTorch JEPA Closed-Loop World Model")
    parser.add_argument("--cycles", type=int, default=3)
    parser.add_argument("--steps", type=int, default=5)
    parser.add_argument("--dt", type=float, default=1.0)
    parser.add_argument("--latent-dim", type=int, default=16)
    parser.add_argument("--hidden-dim", type=int, default=64)
    parser.add_argument("--jepa-weight", type=float, default=0.25)
    parser.add_argument("--temp-c", type=float, default=78.5)
    parser.add_argument("--vib-rms", type=float, default=2.45)
    parser.add_argument("--rate-c-min", type=float, default=1.5)
    parser.add_argument("--load-pct", type=float, default=88.0)
    parser.add_argument("--n-gpu-layers", type=int, default=-1)
    parser.add_argument("--no-resident", action="store_true")
    parser.add_argument("--cache-dir", type=Path, default=None)
    parser.add_argument("--ledger", type=Path, default=Path("world_model_pytorch_jepa_closed_loop_telemetry.jsonl"))
    args, _ = parser.parse_known_args()

    init_s = State(
        temp_c=args.temp_c,
        vib_rms=args.vib_rms,
        rate_c_min=args.rate_c_min,
        load_pct=args.load_pct,
    )
    run_closed_loop(
        initial_state=init_s,
        cycles=args.cycles,
        rollout_steps=args.steps,
        dt=args.dt,
        latent_dim=args.latent_dim,
        hidden_dim=args.hidden_dim,
        jepa_weight=args.jepa_weight,
        n_gpu_layers=args.n_gpu_layers,
        keep_resident=not args.no_resident,
        cache_dir=args.cache_dir,
        ledger=args.ledger,
    )


if __name__ == "__main__":
    run_closed_loop()

llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)
llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)
llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)


SOMALA • CLOSED-LOOP CYCLE 0 TELEMETRY REPORT (PYTORCH JEPA + 5-MODEL GGUF)
Timestamp (UTC) : 2026-11-21T16:29:23.546248+00:00
GPU             : NVIDIA RTX PRO 6000 Blackwell Server Edition
Cycle Latency   : 11.227s
VRAM (GB)       : peak=64.01 GB | post_cycle=64.01 GB
Max Drift       : 0.0

[PyTorch JEPA Latent Engine]
  active=True | device=cuda | dim=16 | weight=0.25 | train_cos_e=0.001534 | z_norm=1.0 | z_in=[0.005, -0.094, 0.048, 0.239]
  selected_cost   : phys_cost=11.9211 + jepa_penalty=0.0005 (avg_e=0.000383) -> cost=11.9216

[State Transition]
  s_0 (in)  : temp_c=78.5 C | vib_rms=2.45 | rate=1.5 C/min | load=88.0%
  s_1 (next): temp_c=78.89 C | vib_rms=2.27 | rate=0.39 C/min | load=68.0%

[Stage Models, SHA-256 & VRAM Load]
  physical  | nemotron-3.5-bolt-topo-governed-Q4_K_M.gguf    | sha=7e59d0116378b6d8 | vram= 4.25 GB (+4.25 GB) | drift=0.0
  perceive  | Gemma-4-26B-A4B-alpaca-topo-epoch-1-Q8_0.gguf  | sha=bfe4b3f85a110f1b | vram=30.28 GB (+26.03 GB) | drift=0.0
  simulat